# GUI-Model: Mobile UI Transition & Action Predictor

12 models × 2 datasets (MobiBench / AndroidControl).

**Stage 1 mode (full / lora)** 은 쉘 스크립트의 `--stage1-mode` 플래그로 선택한다 (default: `full`).
두 모드의 YAML 은 이 노트북에서 모두 생성되며, 산출물 경로·HF Hub ID 는 모드별로 분리된다.


## Dataset Overview

본 프로젝트는 두 가지 모바일 UI 인터랙션 데이터셋을 동시에 처리합니다. Cell 3에서 두 데이터셋 모두 자동 설정됩니다.

### MobiBench

MobiBench 모바일 인터랙션 벤치마크에서 수집된 소규모 데이터셋입니다. 빠른 실험 반복에 적합합니다.

| 항목 | 값 |
|------|-----|
| Stage 1 (World Modeling) | 3,145건 |
| Stage 2 (Action Prediction) | 3,147건 |
| 이미지 | 3,655개 PNG |
| Train/Test Split | 95:5 (seed=42) |

**데이터 형식:**
- **Stage 1**: Screenshot + UI State (XML) + Action → Next UI State (XML)
- **Stage 2**: Screenshot + UI State (XML) + Task Description → Action (JSON)

**Action Type 분포 (Stage 2):**

| Type | 비율 |
|------|------|
| click | ~81.9% |
| input | ~10.6% |
| swipe | ~6.3% |
| long_click | ~0.6% |
| openapp | ~0.6% |

### AndroidControl

AndroidControl 대규모 모바일 제어 데이터셋입니다. MobiBench 대비 Stage 1 ~23배, Stage 2 ~29배 대규모이며, 본 실험의 주력 데이터셋입니다.

| 항목 | 값 |
|------|-----|
| Stage 1 (World Modeling) | 71,047건 |
| Stage 2 (Action Prediction) | 91,677건 |
| Train/Test Split | 95:5 (seed=42) |

**데이터 형식:** MobiBench와 동일한 ShareGPT Multimodal 포맷을 사용합니다.

> **학습 레시피**: 두 데이터셋의 규모 차이를 반영해 MobiBench는 5 epochs(소규모 안정 학습), AndroidControl은 3 epochs(대규모 효율 학습)로 분기됩니다.
> Code2World, gWorld, MobileDreamer 논문 + vlm-gui-finetuning-research.md를 참고하여 설계되었습니다.


## 0. Environment Setup

프로젝트 경로 설정, GPU 확인, 의존성 설치를 수행합니다.

> **`BASE_DIR`을 본인의 프로젝트 경로로 수정하세요.**

In [ ]:
%%bash
pip install -e .

In [ ]:
%%bash
# -----------------------------------------------------------------------------
# [Unsloth 전용 전처리] deepspeed 제거
# -----------------------------------------------------------------------------
# Unsloth (Gemma-4 e2b / e4b) 학습 경로는 deepspeed 와 호환되지 않는다.
#   - Unsloth 의 FastModel/FastVisionModel 은 자체 메모리 최적화 / gradient
#     checkpointing 을 사용하며 deepspeed ZeRO 와 동시 사용 시
#     `RuntimeError: ... deepspeed engine ...` 또는 zero-init weight 가
#     duplicate 되어 학습 첫 step 에서 실패한다.
#   - 동일 env 에 deepspeed 가 설치돼 있으면 `accelerate launch` 가 자동으로
#     deepspeed plugin 을 활성화하려 시도해서 부작용이 발생하기도 한다.
#
# 따라서 backend == "unsloth" 모델을 학습하기 직전에 deepspeed 를 env 에서
# 제거한다. (LlamaFactory 백엔드 모델 학습으로 돌아갈 때는 setup.py 의
# `pip install -e .` 또는 `pip install 'deepspeed>=0.10.0,<=0.18.4'` 로 재설치)
#
# `2>/dev/null || true` 는 deepspeed 가 이미 없을 때도 셀이 빨간색으로
# 멈추지 않도록 한다.
# -----------------------------------------------------------------------------
set -euo pipefail
echo "[before] $(pip show deepspeed 2>/dev/null | awk -F': ' '/^Version/ {print $2}' || echo 'not installed')"
pip uninstall -y deepspeed 2>/dev/null || true
echo "[after ] $(pip show deepspeed 2>/dev/null | awk -F': ' '/^Version/ {print $2}' || echo 'not installed')"

In [ ]:
%%bash
if [ ! -d LlamaFactory ]; then
  git clone https://github.com/hiyouga/LlamaFactory.git
fi

if [ ! -d unsloth ]; then
  git clone https://github.com/unslothai/unsloth.git
fi

In [ ]:
import os
from dotenv import load_dotenv

# ============================================================
# === Project root path ===
# BASE_DIR = "/path/to/GUI-Model"
BASE_DIR = "/home/seungwoo.baek/project/GUI-Model/GUI-Model"
# ============================================================

BASE_DIR = os.path.abspath(BASE_DIR)
LF_ROOT = os.path.join(BASE_DIR, "LlamaFactory")
UNSLOTH_ROOT = os.path.join(BASE_DIR, "unsloth")

load_dotenv(os.path.join(BASE_DIR, ".env"))

# ============================================================
# === Global batch size & NPROC-aware grad_accum ===
# 학습 프레임워크(LlamaFactory / Unsloth)가 서로 다른
# per_device_train_batch_size 를 쓰더라도 effective global batch 가
# GPU 수(NPROC_PER_NODE) 변경에 따라 흔들리지 않도록,
# gradient_accumulation_steps 를 아래 공식으로 자동 역계산한다.
#
#   global_batch = per_device_train_batch_size
#                * gradient_accumulation_steps
#                * NPROC_PER_NODE   (= torchrun world size, single node)
#
#   gradient_accumulation_steps = GLOBAL_BATCH_SIZE
#                                 / (per_device_train_batch_size * NPROC_PER_NODE)
#
# NPROC_PER_NODE 는 .env / 환경변수에서 로드 (scripts/*.sh 와 동일 변수).
# per_device 는 메모리 안정성 기준 프레임워크별 상수 (LF=2, Unsloth=1).
# ============================================================
NPROC_PER_NODE = int(os.getenv("NPROC_PER_NODE", "2"))
GLOBAL_BATCH_SIZE = 64
LF_PER_DEVICE_BS = 2
UNSLOTH_PER_DEVICE_BS = 1


def _derive_grad_accum(per_device: int, nproc: int = NPROC_PER_NODE,
                       target: int = GLOBAL_BATCH_SIZE) -> int:
    """Return gradient_accumulation_steps so that per_device * ga * nproc == target.

    Raise ValueError when target is not divisible by (per_device * nproc); silent
    rounding would alter the effective global batch size.
    """
    denom = per_device * nproc
    if denom <= 0 or target % denom != 0:
        raise ValueError(
            f"GLOBAL_BATCH_SIZE({target}) is not divisible by "
            f"per_device({per_device}) * NPROC_PER_NODE({nproc}) = {denom}. "
            f"Adjust NPROC_PER_NODE in .env or GLOBAL_BATCH_SIZE."
        )
    return target // denom

# ============================================================
# === Model Registry (12 models) ===
# Per-model hparam_overrides tune learning rate / warmup / grad clip /
# weight decay / optimizer / seed / lora_dropout on top of the
# dataset-level baseline (see _DATASET_CONFIG below). Keep epochs,
# image_max_pixels, cutoff_len(=max_seq_length), lora_rank fixed.
# Baseline reference: qwen3-vl-8b on MobiBench.
#
# Stage 1 에는 두 가지 finetuning mode 가 있다:
#   - stage1_full : full fine-tuning (default)
#   - stage1_lora : LoRA fine-tuning  (shell 스크립트 `--stage1-mode lora` 로 선택)
# hparam_overrides 에서 stage1 은 full-mode baseline, stage1_lora 는 그 위에
# 추가로 덮어씌워지는 LoRA 전용 override 이다 (미지정 시 stage1 값 그대로 사용).
#
# Vision / sequence budget (2026-04-18 unified):
#   image_max_pixels = 1,003,520 (= 1280 x 28 x 28), Qwen 공식 표준.
#     -> merged vision tokens ~320 for Qwen / LLaVA.
#     -> Gemma-4 (Unsloth backend) 은 이 필드를 소비하지 않으며
#        HF processor 디폴트 token budget (~280 tokens) 로 동작한다.
#   cutoff_len: stage1=8192 (single-step action),
#               stage2=10000 (multi-step world-model).

# ============================================================
_MODEL_CONFIG = {
    "qwen2-vl-2b": {"model_id": "Qwen/Qwen2-VL-2B-Instruct", "backend": "llamafactory", "short_name": "qwen2-vl-2b", "template": "qwen2_vl", "image_max_pixels": 1003520, "freeze_vision_tower": True, "stage1_deepspeed": "examples/deepspeed/ds_z3_config.json", "stage1_nproc": 4,
        "hparam_overrides": {
            "stage1": {"lr": "1.5e-5", "warmup_ratio": 0.08, "max_grad_norm": 0.5},
            "stage2": {"lr": "5.0e-5", "lora_dropout": 0.05},
        }},
    "qwen2-vl-7b": {"model_id": "Qwen/Qwen2-VL-7B-Instruct", "backend": "llamafactory", "short_name": "qwen2-vl-7b", "template": "qwen2_vl", "image_max_pixels": 1003520, "freeze_vision_tower": True, "stage1_deepspeed": "examples/deepspeed/ds_z3_config.json", "stage1_nproc": 4,
        "hparam_overrides": {}},
    "qwen2.5-vl-3b": {"model_id": "Qwen/Qwen2.5-VL-3B-Instruct", "backend": "llamafactory", "short_name": "qwen2.5-vl-3b", "template": "qwen2_vl", "image_max_pixels": 1003520, "freeze_vision_tower": True, "stage1_deepspeed": "examples/deepspeed/ds_z3_config.json", "stage1_nproc": 4,
        "hparam_overrides": {
            "stage1": {"lr": "1.2e-5", "warmup_ratio": 0.06, "max_grad_norm": 0.5},
            "stage2": {"lr": "4.0e-5"},
        }},
    "qwen2.5-vl-7b": {"model_id": "Qwen/Qwen2.5-VL-7B-Instruct", "backend": "llamafactory", "short_name": "qwen2.5-vl-7b", "template": "qwen2_vl", "image_max_pixels": 1003520, "freeze_vision_tower": True, "stage1_deepspeed": "examples/deepspeed/ds_z3_config.json", "stage1_nproc": 4,
        "hparam_overrides": {}},
    "qwen3-vl-2b": {"model_id": "Qwen/Qwen3-VL-2B-Instruct", "backend": "llamafactory", "short_name": "qwen3-vl-2b", "template": "qwen3_vl_nothink", "image_max_pixels": 1003520, "freeze_vision_tower": True, "stage1_deepspeed": "examples/deepspeed/ds_z3_config.json", "stage1_nproc": 4,
        "hparam_overrides": {
            "stage1": {"lr": "1.5e-5", "warmup_ratio": 0.08, "max_grad_norm": 0.5},
            "stage2": {"lr": "5.0e-5", "lora_dropout": 0.05},
        }},
    "qwen3-vl-4b": {"model_id": "Qwen/Qwen3-VL-4B-Instruct", "backend": "llamafactory", "short_name": "qwen3-vl-4b", "template": "qwen3_vl_nothink", "image_max_pixels": 1003520, "freeze_vision_tower": True, "stage1_deepspeed": "examples/deepspeed/ds_z3_config.json", "stage1_nproc": 4,
        "hparam_overrides": {
            "stage1": {"lr": "1.2e-5", "warmup_ratio": 0.06},
            "stage2": {"lr": "4.0e-5"},
        }},
    "qwen3-vl-8b": {"model_id": "Qwen/Qwen3-VL-8B-Instruct", "backend": "llamafactory", "short_name": "qwen3-vl-8b", "template": "qwen3_vl_nothink", "image_max_pixels": 1003520, "freeze_vision_tower": True, "stage1_deepspeed": "examples/deepspeed/ds_z3_config.json", "stage1_nproc": 4,
        "hparam_overrides": {}},
    "gemma-4-e2b": {"model_id": "unsloth/gemma-4-E2B-it", "backend": "unsloth", "short_name": "gemma-4-e2b", "template": "gemma-4", "image_max_pixels": 1003520, "freeze_vision_tower": True, "stage1_deepspeed": "examples/deepspeed/ds_z3_config.json", "stage1_nproc": 4,
        "hparam_overrides": {
            "stage1": {"lr": "2.0e-5", "warmup_ratio": 0.1, "optim": "adamw_torch_fused", "seed": 3407},
            "stage1_lora": {"lr": "1.0e-4"},
            "stage2": {"lr": "5.0e-5", "lora_dropout": 0.05, "optim": "adamw_torch_fused", "seed": 3407},
        }},
    "gemma-4-e4b": {"model_id": "unsloth/gemma-4-E4B-it", "backend": "unsloth", "short_name": "gemma-4-e4b", "template": "gemma-4", "image_max_pixels": 1003520, "freeze_vision_tower": True, "stage1_deepspeed": "examples/deepspeed/ds_z3_config.json", "stage1_nproc": 4,
        "hparam_overrides": {
            "stage1": {"lr": "2.0e-5", "warmup_ratio": 0.1, "optim": "adamw_torch_fused", "seed": 3407},
            "stage1_lora": {"lr": "1.0e-4"},
            "stage2": {"lr": "4.0e-5", "optim": "adamw_torch_fused", "seed": 3407},
        }},
    "llava-v1.6-mistral-7b": {"model_id": "llava-hf/llava-v1.6-mistral-7b-hf", "backend": "llamafactory", "short_name": "llava-v1.6-mistral-7b", "template": "llava_next", "image_max_pixels": 1003520, "freeze_vision_tower": True, "stage1_deepspeed": "examples/deepspeed/ds_z3_config.json", "stage1_nproc": 4,
        "hparam_overrides": {
            "stage1": {"lr": "2.0e-5", "warmup_ratio": 0.03, "weight_decay": 0.0},
            "stage2": {"lr": "4.0e-5", "warmup_ratio": 0.03, "weight_decay": 0.0},
        }},
    "llava-v1.6-vicuna-7b": {"model_id": "llava-hf/llava-v1.6-vicuna-7b-hf", "backend": "llamafactory", "short_name": "llava-v1.6-vicuna-7b", "template": "llava_next", "image_max_pixels": 1003520, "freeze_vision_tower": True, "stage1_deepspeed": "examples/deepspeed/ds_z3_config.json", "stage1_nproc": 4,
        "hparam_overrides": {
            "stage1": {"lr": "1.5e-5", "warmup_ratio": 0.03, "weight_decay": 0.0},
            "stage2": {"lr": "3.0e-5", "warmup_ratio": 0.03, "weight_decay": 0.0},
        }},
    "llama3-llava-next-8b": {"model_id": "llava-hf/llama3-llava-next-8b-hf", "backend": "llamafactory", "short_name": "llama3-llava-next-8b", "template": "llava_next", "image_max_pixels": 1003520, "freeze_vision_tower": True, "stage1_deepspeed": "examples/deepspeed/ds_z3_config.json", "stage1_nproc": 4,
        "hparam_overrides": {
            "stage1": {"lr": "1.5e-5", "warmup_ratio": 0.03, "weight_decay": 0.0},
            "stage2": {"lr": "3.0e-5", "warmup_ratio": 0.03, "weight_decay": 0.0},
        }},
}

# ============================================================
# === Dataset configs (baseline hparams per dataset) ===
# stage1 에는 LoRA 기본 hparams (lora_rank/alpha/dropout) 가 포함된다.
# LoRA mode 에서만 소비되고 full mode 에서는 무시된다.
# Per-model overrides 는 _MODEL_CONFIG["hparam_overrides"] 로 덮어씀:
#   - hparam_overrides["stage1"]      → full-mode baseline override
#   - hparam_overrides["stage1_lora"] → LoRA-mode 추가 override
#   - hparam_overrides["stage2"]      → stage2 (항상 LoRA) override
# gradient_accumulation_steps 는 _DATASET_CONFIG 에 두지 않는다.
# NPROC_PER_NODE 와 프레임워크별 per_device 에 따라 CONFIGS 빌드 시
# _derive_grad_accum() 으로 자동 계산되어 stage dict 에 주입된다.
# ============================================================
_DATASET_CONFIG = {
    "MobiBench": {
        "lf_subfolder": "GUI-Model-MB",
        "ds_prefix": "GUI-Model-MB",
        "output_prefix": "MB/",
        "hf_slug": "mb-",
        "stage1": {
            "lr": "1.0e-5", "epochs": 5, "warmup_ratio": 0.05,
            "save_strategy": "epoch", "save_steps": None,
            "eval_strategy": "epoch", "eval_steps": None,
            "per_device_eval_batch_size": 1,
            "lora_rank": 8, "lora_alpha": 16, "lora_dropout": 0.05,
            "weight_decay": 0.01, "max_grad_norm": 1.0,
            "lr_scheduler_type": "cosine",
        },
        "stage2": {
            "lr": "3.0e-5", "epochs": 5, "warmup_ratio": 0.05,
            "save_strategy": "epoch", "save_steps": None,
            "eval_strategy": "epoch", "eval_steps": None,
            "per_device_eval_batch_size": 1,
            "lora_rank": 16, "lora_alpha": 32, "lora_dropout": 0.1,
            "weight_decay": 0.01, "max_grad_norm": 1.0,
            "lr_scheduler_type": "cosine",
        },
    },
    "AndroidControl": {
        "lf_subfolder": "GUI-Model-AC",
        "ds_prefix": "GUI-Model-AC",
        "output_prefix": "AC/",
        "hf_slug": "ac-",
        "stage1": {
            "lr": "1.0e-5", "epochs": 3, "warmup_ratio": 0.03,
            "save_strategy": "epoch", "save_steps": None,
            "eval_strategy": "epoch", "eval_steps": None,
            "per_device_eval_batch_size": 4,
            "lora_rank": 8, "lora_alpha": 16, "lora_dropout": 0.05,
            "weight_decay": 0.01, "max_grad_norm": 1.0,
            "lr_scheduler_type": "cosine",
        },
        "stage2": {
            "lr": "5.0e-5", "epochs": 3, "warmup_ratio": 0.03,
            "save_strategy": "epoch", "save_steps": None,
            "eval_strategy": "epoch", "eval_steps": None,
            "per_device_eval_batch_size": 4,
            "lora_rank": 32, "lora_alpha": 64, "lora_dropout": 0.1,
            "weight_decay": 0.01, "max_grad_norm": 1.0,
            "lr_scheduler_type": "cosine",
        },
    },
}

# ============================================================
# === Model ordering for execution cells ===
# ============================================================
MODEL_ORDER = [
    ("qwen2-vl-2b", "Qwen2-VL-2B"),
    ("qwen2-vl-7b", "Qwen2-VL-7B"),
    ("qwen2.5-vl-3b", "Qwen2.5-VL-3B"),
    ("qwen2.5-vl-7b", "Qwen2.5-VL-7B"),
    ("qwen3-vl-2b", "Qwen3-VL-2B"),
    ("qwen3-vl-4b", "Qwen3-VL-4B"),
    ("qwen3-vl-8b", "Qwen3-VL-8B"),
    ("gemma-4-e2b", "Gemma-4-E2B"),
    ("gemma-4-e4b", "Gemma-4-E4B"),
    ("llava-v1.6-mistral-7b", "LLaVA-v1.6-Mistral-7B"),
    ("llava-v1.6-vicuna-7b", "LLaVA-v1.6-Vicuna-7B"),
    ("llama3-llava-next-8b", "LLaMA3-LLaVA-Next-8B"),
]
DS_ORDER = [("MB", "MobiBench"), ("AC", "AndroidControl")]

# ============================================================
# === Build CONFIGS: CONFIGS[model_key][ds_name] ===
# Stage 1 은 full / lora 두 벌이 공존할 수 있도록 경로 · Hub ID · hparams
# 모두 `_full` / `_lora` 접미사로 분리해 저장한다. 레거시 키
#   save_s1, out_s1_merged, hf_s1_model, stage1
# 는 full 버전의 alias 로 유지한다 (기존 실행 셀 호환).
#
# Stage 2 world-model variant 도 상류 Stage 1 모드에 따라 2 벌 생성한다:
#   save_s2_world_from_full      / save_s2_world_from_lora
#   out_s2_merged_world_from_full/ out_s2_merged_world_from_lora
#   hf_s2_world_full             / hf_s2_world_lora
# 레거시 키 save_s2_world, out_s2_merged_world, hf_s2_world 는 from_full 의 alias.
# ============================================================
CONFIGS = {}
for _model_key, _mcfg in _MODEL_CONFIG.items():
    CONFIGS[_model_key] = {}
    _overrides = _mcfg.get("hparam_overrides", {})
    _backend = _mcfg.get("backend", "llamafactory")
    _per_device = UNSLOTH_PER_DEVICE_BS if _backend == "unsloth" else LF_PER_DEVICE_BS
    _grad_accum = _derive_grad_accum(_per_device)
    for _ds_name, _cfg in _DATASET_CONFIG.items():
        c = dict(_cfg)
        c["model_key"] = _model_key
        c["model_id"] = _mcfg["model_id"]
        c["short_name"] = _mcfg["short_name"]
        c["template"] = _mcfg["template"]
        c["model_config"] = _mcfg
        c["dataset_name"] = _ds_name
        c["data_dir"] = os.path.join(BASE_DIR, "data", _ds_name)
        c["ds_s1_train"] = f"{c['ds_prefix']}_stage1_train"
        c["ds_s1_test"] = f"{c['ds_prefix']}_stage1_test"
        c["ds_s2_train"] = f"{c['ds_prefix']}_stage2_train"
        c["ds_s2_test"] = f"{c['ds_prefix']}_stage2_test"

        # Stage 1 HF Hub IDs — mode-tagged (full / lora)
        c["hf_s1_model_full"] = f"SaFD-00/{_mcfg['short_name']}-{c['hf_slug']}stage1-full-world-model"
        c["hf_s1_model_lora"] = f"SaFD-00/{_mcfg['short_name']}-{c['hf_slug']}stage1-lora-world-model"
        c["hf_s1_model"] = c["hf_s1_model_full"]  # legacy alias → default (full)

        # Stage 2 HF Hub IDs — base is S1-mode 무관, world-model 은 상류 S1 모드에 따라 분리
        c["hf_s2_base"] = f"SaFD-00/{_mcfg['short_name']}-{c['hf_slug']}stage2-base"
        c["hf_s2_world_full"] = f"SaFD-00/{_mcfg['short_name']}-{c['hf_slug']}stage2-full-world-model"
        c["hf_s2_world_lora"] = f"SaFD-00/{_mcfg['short_name']}-{c['hf_slug']}stage2-lora-world-model"
        c["hf_s2_world"] = c["hf_s2_world_full"]  # legacy alias

        # 저장 경로: GUI-Model/outputs/{DS}/{category}/{MODEL}_{detail}/... (flat naming).
        # LF 는 cwd=LF_ROOT, Unsloth 는 cwd=UNSLOTH_ROOT. 두 백엔드 모두 각자 ROOT 기준
        # "../outputs/", "../data/" 상대경로를 사용 → 최종 물리 경로는 GUI-Model/outputs/ 로 동일하게 수렴.
        _ds_code = c["output_prefix"].rstrip("/")  # "MB" | "AC"
        _mshort  = _mcfg["short_name"]

        # Stage 1 adapter / merged dirs — mode-tagged
        c["save_s1_full"]        = f"../outputs/{_ds_code}/adapters/{_mshort}_stage1_full"
        c["save_s1_lora"]        = f"../outputs/{_ds_code}/adapters/{_mshort}_stage1_lora"
        c["out_s1_merged_full"]  = f"../outputs/{_ds_code}/merged/{_mshort}_stage1_full"
        c["out_s1_merged_lora"]  = f"../outputs/{_ds_code}/merged/{_mshort}_stage1_lora"
        c["save_s1"]        = c["save_s1_full"]        # legacy alias
        c["out_s1_merged"]  = c["out_s1_merged_full"]  # legacy alias

        # Stage 2 adapter / merged dirs
        c["save_s2_base"]                 = f"../outputs/{_ds_code}/adapters/{_mshort}_stage2_lora_base"
        c["save_s2_world_from_full"]      = f"../outputs/{_ds_code}/adapters/{_mshort}_stage2_lora_world_model_from_full"
        c["save_s2_world_from_lora"]      = f"../outputs/{_ds_code}/adapters/{_mshort}_stage2_lora_world_model_from_lora"
        c["out_s2_merged_base"]           = f"../outputs/{_ds_code}/merged/{_mshort}_stage2_lora_base"
        c["out_s2_merged_world_from_full"]= f"../outputs/{_ds_code}/merged/{_mshort}_stage2_lora_world_model_from_full"
        c["out_s2_merged_world_from_lora"]= f"../outputs/{_ds_code}/merged/{_mshort}_stage2_lora_world_model_from_lora"
        c["save_s2_world"]        = c["save_s2_world_from_full"]         # legacy alias
        c["out_s2_merged_world"]  = c["out_s2_merged_world_from_full"]   # legacy alias

        # ---- Merge per-model hparam overrides onto dataset baseline ----
        # Stage 1 full: baseline ∪ stage1 override
        s1_full = dict(c["stage1"]); s1_full.update(_overrides.get("stage1", {}))
        # Stage 1 lora: stage1_full ∪ stage1_lora override (LoRA 전용 덮어쓰기)
        s1_lora = dict(s1_full);     s1_lora.update(_overrides.get("stage1_lora", {}))
        s2 = dict(c["stage2"]); s2.update(_overrides.get("stage2", {}))

        # === Auto-derive grad_accum so that effective global batch == GLOBAL_BATCH_SIZE ===
        # backend 별 per_device 는 Cell 9/9b/11/11b/15/17 의 YAML f-string 에 하드코딩된 값과 일치해야 한다.
        s1_full["gradient_accumulation_steps"] = _grad_accum
        s1_lora["gradient_accumulation_steps"] = _grad_accum
        s2["gradient_accumulation_steps"] = _grad_accum

        c["stage1_full"] = s1_full
        c["stage1_lora"] = s1_lora
        c["stage1"]      = s1_full   # legacy alias → default (full)
        c["stage2"]      = s2
        c["per_device_train_batch_size"] = _per_device  # 디버깅 편의

        CONFIGS[_model_key][_ds_name] = c

print(f"Working directory: {os.getcwd()}")
print(f"LlamaFactory root: {LF_ROOT}")
print(f"Unsloth root: {UNSLOTH_ROOT}")
print(f"NPROC_PER_NODE={NPROC_PER_NODE}, GLOBAL_BATCH_SIZE={GLOBAL_BATCH_SIZE}")
print(f"  LF      : per_device={LF_PER_DEVICE_BS}, grad_accum={_derive_grad_accum(LF_PER_DEVICE_BS)}  "
      f"(global = {LF_PER_DEVICE_BS} * {_derive_grad_accum(LF_PER_DEVICE_BS)} * {NPROC_PER_NODE} = {GLOBAL_BATCH_SIZE})")
print(f"  Unsloth : per_device={UNSLOTH_PER_DEVICE_BS}, grad_accum={_derive_grad_accum(UNSLOTH_PER_DEVICE_BS)}  "
      f"(global = {UNSLOTH_PER_DEVICE_BS} * {_derive_grad_accum(UNSLOTH_PER_DEVICE_BS)} * {NPROC_PER_NODE} = {GLOBAL_BATCH_SIZE})")
print(f"Models: {list(CONFIGS.keys())}")
print(f"Datasets: {list(_DATASET_CONFIG.keys())}")
for _mk in list(CONFIGS.keys())[:2]:
    for _ds, c in CONFIGS[_mk].items():
        s1f, s1l, s2 = c["stage1_full"], c["stage1_lora"], c["stage2"]
        print(f"\n--- {_mk} / {_ds} ---")
        print(f"  model_id: {c['model_id']}")
        print(f"  template: {c['template']}")
        print(f"  stage1 FULL lr/warmup/wd/grad_norm: {s1f['lr']}/{s1f['warmup_ratio']}/{s1f['weight_decay']}/{s1f['max_grad_norm']}")
        print(f"  stage1 LORA lr / lora_rank/alpha/dropout:  {s1l['lr']} / r={s1l['lora_rank']}/a={s1l['lora_alpha']}/d={s1l['lora_dropout']}")
        print(f"  stage2 lr/warmup/lora_dropout:  {s2['lr']}/{s2['warmup_ratio']}/{s2['lora_dropout']}")
        print(f"  save_s1_full: {c['save_s1_full']}")
        print(f"  save_s1_lora: {c['save_s1_lora']}")
        print(f"  hf_s1_model_full: {c['hf_s1_model_full']}")
        print(f"  hf_s1_model_lora: {c['hf_s1_model_lora']}")
print("\n... (remaining models omitted)")


### YAML Configs 일괄 생성

이 블록은 `LlamaFactory/examples/custom/` 하위 YAML 과 `unsloth/configs/` 하위 YAML 을 **환경설정 직후 한 번에 생성**합니다. 이후 학습·평가 단계에서 별도의 YAML 생성 셀 실행 없이 즉시 스크립트/CLI 를 호출할 수 있습니다.

- **`[LlamaFactory]` 블록** — `backend == "llamafactory"` 모델(qwen / llava 계열)만 처리. cwd = `LF_ROOT`, 경로는 `../outputs/` 기준.
- **`[Unsloth]` 블록** — `backend == "unsloth"` 모델(gemma-4-e2b / e4b)만 처리. cwd = `UNSLOTH_ROOT`, 경로는 `../outputs/`, `../data/` 기준.

**포함 대상 (12개 모델 × 2개 데이터셋)**

| 블록 | 생성 파일 | 비고 |
|------|-----------|------|
| `[LlamaFactory]` Stage 1 Training YAML | `custom/GUI-Model-{MB,AC}/stage1_full/{MODEL}_world-model.yaml` | Full FT, output → `outputs/{DS}/adapters/{MODEL}_stage1_full/` |
| `[Unsloth]` Stage 1 Full YAML | `unsloth/configs/GUI-Model-{MB,AC}/stage1_full/{MODEL}_world-model.yaml` | gemma-4-e2b / e4b 전용, output → 동일 flat 경로 |
| `[LlamaFactory]` Stage 2 Training YAML | `custom/GUI-Model-{MB,AC}/stage2_lora/{MODEL}_{base,world-model}.yaml` | LoRA 2 variants |
| `[Unsloth]` Stage 2 LoRA YAML | `unsloth/configs/GUI-Model-{MB,AC}/stage2_lora/{MODEL}_{base,world-model}.yaml` | gemma-4-e2b / e4b 전용 |

**Merge YAML** (Section 5 · Section 8 에서 별도 생성)

| 블록 | 생성 파일 | 비고 |
|------|-----------|------|
| `[LlamaFactory]` Stage 1 Merge YAML | `custom/GUI-Model-{MB,AC}/stage1_merge/{MODEL}_world-model.yaml` | `outputs/{DS}/adapters/{MODEL}_stage1_full/BEST_CHECKPOINT` winner → `outputs/{DS}/merged/{MODEL}_stage1_full/` |
| `[LlamaFactory]` Stage 2 Merge YAML | `custom/GUI-Model-{MB,AC}/stage2_merge/{MODEL}_{base,world-model}.yaml` | `outputs/{DS}/adapters/{MODEL}_stage2_lora_{base,world_model}/BEST_CHECKPOINT` winner → `outputs/{DS}/merged/{MODEL}_stage2_lora_{base,world_model}/` |

> Unsloth backend 의 eval/merge 는 YAML 이 아닌 shell 스크립트 + CLI 인자(`scripts/_unsloth_merge.py`) 로 구동되므로 별도 생성 셀이 없습니다.

> `{MODEL}` = 모델 short_name (예: `qwen3-vl-8b`, `gemma-4-e2b` 등 12개). `{DS}` = `MB` 또는 `AC`. 저장 경로는 **flat 네이밍** (`{MODEL}_{detail}/`) 을 사용합니다 (eval 만 중첩 구조 유지).

**학습 중 eval 비활성화**: Stage 1/2 학습 YAML 의 `### eval` 블록은 주석 처리되어 있어 학습 시 eval 이 실행되지 않습니다. Best checkpoint 선택은 학습 완료 후 `stage{1,2}_eval.sh` (Hungarian F1 / Overall Score 기반 sweep) 가 담당합니다.

> **Merge YAML 재실행 필요**: Merge YAML 은 `BEST_CHECKPOINT` 파일을 읽어 winner 경로를 주입합니다. **이 파일은 평가 완료 후에야 생성**되므로, 최초 생성 시 `BEST_CHECKPOINT` 가 아직 없는 모델은 `[SKIP]` 경고와 함께 건너뜁니다(정상 동작). 평가가 끝난 뒤 해당 Merge YAML 셀을 **재실행**하면 건너뛰었던 항목의 Merge YAML 이 추가 생성됩니다.


#### [LlamaFactory] Stage 1 Training YAML

`LlamaFactory/examples/custom/GUI-Model-{MB,AC}/stage1_full/{MODEL}_world-model.yaml` 생성. LF_ROOT 기준 상대경로 (`../outputs/`) 사용. `backend == "llamafactory"` 모델만 대상 (qwen / llava 계열).


In [ ]:
from pathlib import Path

# LF Stage 1 YAML — full / lora 두 벌 생성.
# LoRA mode 는 finetuning_type: lora + lora_rank/alpha/target/dropout 블록을 추가.
for MODE in ["full", "lora"]:
    for model_key, ds_configs in CONFIGS.items():
        for ds_name, cfg in ds_configs.items():
            if cfg["model_config"].get("backend") == "unsloth":
                continue
            s1 = cfg[f"stage1_{MODE}"]
            mcfg = cfg["model_config"]
            yaml_dir = Path(LF_ROOT) / "examples" / "custom" / cfg["lf_subfolder"] / f"stage1_{MODE}"
            yaml_dir.mkdir(parents=True, exist_ok=True)

            save_steps_line = f"save_steps: {s1['save_steps']}\n" if s1["save_strategy"] == "steps" else ""
            ds_line = f"deepspeed: {mcfg['stage1_deepspeed']}\n" if mcfg.get("stage1_deepspeed") else ""
            optim_line = f"optim: {s1['optim']}\n" if s1.get("optim") else ""
            seed_line = f"seed: {s1['seed']}\n" if s1.get("seed") is not None else ""

            output_dir = cfg[f"save_s1_{MODE}"]

            if MODE == "full":
                method_block = f"""\
### method
stage: sft
do_train: true
finetuning_type: full
freeze_vision_tower: {str(mcfg["freeze_vision_tower"]).lower()}"""
            else:
                method_block = f"""\
### method
stage: sft
do_train: true
finetuning_type: lora
freeze_vision_tower: {str(mcfg["freeze_vision_tower"]).lower()}
lora_rank: {s1["lora_rank"]}
lora_alpha: {s1["lora_alpha"]}
lora_target: all
lora_dropout: {s1["lora_dropout"]}"""

            STAGE1_CONFIG = f"""\
### model
model_name_or_path: {cfg["model_id"]}
trust_remote_code: true
image_max_pixels: {mcfg["image_max_pixels"]}

{method_block}

### dataset
dataset: {cfg["ds_s1_train"]}
template: {cfg["template"]}
cutoff_len: 8192
overwrite_cache: false
preprocessing_num_workers: 16

### output
output_dir: {output_dir}
logging_steps: 1
save_strategy: {s1["save_strategy"]}
{save_steps_line}save_total_limit: 5
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: 2
gradient_accumulation_steps: {s1["gradient_accumulation_steps"]}
learning_rate: {s1["lr"]}
num_train_epochs: {s1["epochs"]}
lr_scheduler_type: {s1["lr_scheduler_type"]}
warmup_ratio: {s1["warmup_ratio"]}
weight_decay: {s1["weight_decay"]}
max_grad_norm: {s1["max_grad_norm"]}
{optim_line}{seed_line}bf16: true
gradient_checkpointing: true
{ds_line}ddp_timeout: 18000000
# resume_from_checkpoint: true
"""

            fpath = yaml_dir / f"{model_key}_world-model.yaml"
            with open(fpath, 'w') as f:
                f.write(STAGE1_CONFIG)
            print(f"[LF/{MODE}] {fpath.relative_to(Path(LF_ROOT))}")
            print(f"  model: {cfg['model_id']}, template: {cfg['template']}")
            print(f"  lr={s1['lr']}, warmup={s1['warmup_ratio']}, wd={s1['weight_decay']}, grad_norm={s1['max_grad_norm']}")
            if MODE == "lora":
                print(f"  lora: r={s1['lora_rank']}, alpha={s1['lora_alpha']}, dropout={s1['lora_dropout']}")
            print(f"  output_dir: {output_dir}")
            print()


#### [Unsloth] Stage 1 Full YAML (gemma-4-e2b / gemma-4-e4b)

`unsloth/configs/GUI-Model-{MB,AC}/stage1_full/{MODEL}_world-model.yaml` 생성. UNSLOTH_ROOT 기준 상대경로 (`../outputs/`, `../data/`) 사용. `backend == "unsloth"` 모델만 대상 (gemma-4 계열).


In [ ]:
from pathlib import Path

# Unsloth Stage 1 YAML — full / lora 두 벌 생성 (Gemma-4 계열).
for MODE in ["full", "lora"]:
    for model_key, ds_configs in CONFIGS.items():
        _mcfg = next(iter(ds_configs.values()))["model_config"]
        if _mcfg.get("backend") != "unsloth":
            continue
        for ds_name, cfg in ds_configs.items():
            s1 = cfg[f"stage1_{MODE}"]
            _ds_code = cfg["output_prefix"].rstrip("/")   # "MB" | "AC"
            yaml_dir = Path(UNSLOTH_ROOT) / "configs" / f"GUI-Model-{_ds_code}" / f"stage1_{MODE}"
            yaml_dir.mkdir(parents=True, exist_ok=True)

            output_dir = cfg[f"save_s1_{MODE}"]

            if MODE == "full":
                method_block = f"""\
### method
stage: sft
do_train: true
finetuning_type: full
full_finetuning: true
freeze_vision_tower: {str(_mcfg["freeze_vision_tower"]).lower()}"""
            else:
                method_block = f"""\
### method
stage: sft
do_train: true
finetuning_type: lora
freeze_vision_tower: {str(_mcfg["freeze_vision_tower"]).lower()}

### lora
lora_rank: {s1["lora_rank"]}
lora_alpha: {s1["lora_alpha"]}
lora_target: all
lora_dropout: {s1["lora_dropout"]}"""

            # full 모드에서만 load_in_16bit: true 사용 (기존 동작 유지).
            bit_line = "load_in_16bit: true" if MODE == "full" else "load_in_4bit: false"

            UNSLOTH_STAGE1_CONFIG = f"""\
### model
model_name_or_path: {cfg["model_id"]}
max_seq_length: 8192
load_in_4bit: false
{bit_line if MODE == "full" else ""}

{method_block}

### dataset
dataset_path: ../data/{ds_name}/gui-model_stage1_train.jsonl
image_base_dir: ../data/{ds_name}
template: {cfg["template"]}
cutoff_len: 8192

### output
output_dir: {output_dir}
logging_steps: 1
save_strategy: {s1["save_strategy"]}
save_total_limit: 5
overwrite_output_dir: true

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: {s1["gradient_accumulation_steps"]}
learning_rate: {s1["lr"]}
num_train_epochs: {s1["epochs"]}
lr_scheduler_type: {s1["lr_scheduler_type"]}
warmup_ratio: {s1["warmup_ratio"]}
weight_decay: {s1["weight_decay"]}
max_grad_norm: {s1["max_grad_norm"]}
optim: {s1.get("optim", "adamw_torch_fused")}
bf16: true
gradient_checkpointing: unsloth
ddp_find_unused_parameters: true
seed: 3407
"""

            fpath = yaml_dir / f"{model_key}_world-model.yaml"
            with open(fpath, "w") as f:
                f.write(UNSLOTH_STAGE1_CONFIG)
            print(f"[Unsloth/{MODE}] {fpath.relative_to(Path(BASE_DIR))}")
            print(f"  model: {cfg['model_id']}, template: {cfg['template']}")
            if MODE == "lora":
                print(f"  lora: r={s1['lora_rank']}, alpha={s1['lora_alpha']}, dropout={s1['lora_dropout']}")
            print(f"  output_dir: {output_dir}")
            print()


#### [LlamaFactory] Stage 2 Training YAML

`LlamaFactory/examples/custom/GUI-Model-{MB,AC}/stage2_lora/{MODEL}_{base,world-model}.yaml` 생성. world_model 은 Stage1 merged 모델(HF Hub)을 base 로 사용. LF_ROOT 기준 상대경로 (`../outputs/`) 사용. `backend == "llamafactory"` 모델만 대상 (qwen / llava 계열).


In [ ]:
import os
from pathlib import Path

# Stage 2 는 항상 LoRA. world-model variant 는 상류 Stage 1 모드에 따라
# from-full / from-lora 두 벌 생성 (옵션 A — pre-generate both).
for model_key, ds_configs in CONFIGS.items():
    for ds_name, cfg in ds_configs.items():
        if cfg["model_config"].get("backend") == "unsloth":
            continue
        s2 = cfg["stage2"]
        mcfg = cfg["model_config"]
        yaml_dir = Path(LF_ROOT) / "examples" / "custom" / cfg["lf_subfolder"] / "stage2_lora"
        yaml_dir.mkdir(parents=True, exist_ok=True)

        save_steps_line = f"save_steps: {s2['save_steps']}\n" if s2["save_strategy"] == "steps" else ""
        optim_line = f"optim: {s2['optim']}\n" if s2.get("optim") else ""
        seed_line = f"seed: {s2['seed']}\n" if s2.get("seed") is not None else ""

        COMMON_CONFIG = f"""\
### model
model_name_or_path: {{model_name_or_path}}
trust_remote_code: true
image_max_pixels: {mcfg["image_max_pixels"]}

### method
stage: sft
do_train: true
finetuning_type: lora
freeze_vision_tower: {str(mcfg["freeze_vision_tower"]).lower()}
lora_rank: {s2["lora_rank"]}
lora_alpha: {s2["lora_alpha"]}
lora_target: all
lora_dropout: {s2["lora_dropout"]}

### dataset
dataset: {cfg["ds_s2_train"]}
template: {cfg["template"]}
cutoff_len: 10000
overwrite_cache: false
preprocessing_num_workers: 16

### output
output_dir: {{output_dir}}
logging_steps: 1
save_strategy: {s2["save_strategy"]}
{save_steps_line}save_total_limit: 5
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: 2
gradient_accumulation_steps: {s2["gradient_accumulation_steps"]}
learning_rate: {s2["lr"]}
num_train_epochs: {s2["epochs"]}
lr_scheduler_type: {s2["lr_scheduler_type"]}
warmup_ratio: {s2["warmup_ratio"]}
weight_decay: {s2["weight_decay"]}
max_grad_norm: {s2["max_grad_norm"]}
{optim_line}{seed_line}bf16: true
gradient_checkpointing: true
ddp_timeout: 18000000
# resume_from_checkpoint: true
"""

        VARIANTS = {
            "base": {
                "model_name_or_path": cfg["model_id"],
                "output_dir": cfg["save_s2_base"],
            },
            "world-model-full": {
                "model_name_or_path": cfg["hf_s1_model_full"],
                "output_dir": cfg["save_s2_world_from_full"],
            },
            "world-model-lora": {
                "model_name_or_path": cfg["hf_s1_model_lora"],
                "output_dir": cfg["save_s2_world_from_lora"],
            },
        }

        print(f"=== {model_key} / {ds_name} Stage 2 YAML ===")
        print(f"  lr={s2['lr']}, warmup={s2['warmup_ratio']}, wd={s2['weight_decay']}, "
              f"grad_norm={s2['max_grad_norm']}, lora_alpha={s2['lora_alpha']}, lora_dropout={s2['lora_dropout']}")
        for variant, params in VARIANTS.items():
            content = COMMON_CONFIG.format(
                model_name_or_path=params["model_name_or_path"],
                output_dir=params["output_dir"],
            )
            fpath = yaml_dir / f"{model_key}_{variant}.yaml"
            with open(fpath, 'w') as f:
                f.write(content)
            print(f"[Created] {fpath.relative_to(Path(LF_ROOT))}")
            print(f"  base:   {params['model_name_or_path']}")
            print(f"  output: {params['output_dir']}")
            print()


#### [Unsloth] Stage 2 LoRA YAML (base + world_model × gemma-4-e2b / gemma-4-e4b)

`unsloth/configs/GUI-Model-{MB,AC}/stage2_lora/{MODEL}_{base,world-model}.yaml` 생성. world_model 은 Stage1 merged 모델(HF Hub)을 base 로 사용. UNSLOTH_ROOT 기준 상대경로 (`../outputs/`, `../data/`) 사용. `backend == "unsloth"` 모델만 대상 (gemma-4 계열).


In [ ]:
from pathlib import Path

# Unsloth Stage 2 YAML — base + world-model-full + world-model-lora 3 variants.
for model_key, ds_configs in CONFIGS.items():
    _mcfg = next(iter(ds_configs.values()))["model_config"]
    if _mcfg.get("backend") != "unsloth":
        continue
    for ds_name, cfg in ds_configs.items():
        s2 = cfg["stage2"]
        _ds_code = cfg["output_prefix"].rstrip("/")
        yaml_dir = Path(UNSLOTH_ROOT) / "configs" / f"GUI-Model-{_ds_code}" / "stage2_lora"
        yaml_dir.mkdir(parents=True, exist_ok=True)

        variants = [
            ("base",
             cfg["save_s2_base"],
             cfg["model_id"],
             ""),
            ("world-model-full",
             cfg["save_s2_world_from_full"],
             cfg["hf_s1_model_full"],
             f"# Stage 1 merged (full) — HF Hub {cfg['hf_s1_model_full']} or local {cfg['out_s1_merged_full'][3:] if cfg['out_s1_merged_full'].startswith('../') else cfg['out_s1_merged_full']}\n"),
            ("world-model-lora",
             cfg["save_s2_world_from_lora"],
             cfg["hf_s1_model_lora"],
             f"# Stage 1 merged (lora) — HF Hub {cfg['hf_s1_model_lora']} or local {cfg['out_s1_merged_lora'][3:] if cfg['out_s1_merged_lora'].startswith('../') else cfg['out_s1_merged_lora']}\n"),
        ]

        for variant_name, out_path, model_ref, head_comment in variants:
            UNSLOTH_STAGE2_CONFIG = f"""\
### model
{head_comment}model_name_or_path: {model_ref}
max_seq_length: 10000
load_in_4bit: false

### method
stage: sft
do_train: true
finetuning_type: lora
freeze_vision_tower: {str(_mcfg["freeze_vision_tower"]).lower()}

### lora
lora_rank: {s2["lora_rank"]}
lora_alpha: {s2["lora_alpha"]}
lora_target: all
lora_dropout: {s2["lora_dropout"]}

### dataset
dataset_path: ../data/{ds_name}/gui-model_stage2_train.jsonl
image_base_dir: ../data/{ds_name}
template: {cfg["template"]}
cutoff_len: 10000

### output
output_dir: {out_path}
logging_steps: 1
save_strategy: {s2["save_strategy"]}
save_total_limit: 5
overwrite_output_dir: true

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: {s2["gradient_accumulation_steps"]}
learning_rate: {s2["lr"]}
num_train_epochs: {s2["epochs"]}
lr_scheduler_type: {s2["lr_scheduler_type"]}
warmup_ratio: {s2["warmup_ratio"]}
weight_decay: {s2["weight_decay"]}
max_grad_norm: {s2["max_grad_norm"]}
optim: {s2.get("optim", "adamw_torch_fused")}
bf16: true
gradient_checkpointing: unsloth
ddp_find_unused_parameters: true
seed: 3407
"""
            fpath = yaml_dir / f"{model_key}_{variant_name}.yaml"
            with open(fpath, "w") as f:
                f.write(UNSLOTH_STAGE2_CONFIG)
            print(f"[Unsloth] {fpath.relative_to(Path(BASE_DIR))}")
            print(f"  base:   {model_ref}")
            print(f"  output: {out_path}")
            print()


## 1. Stage 1 Data Preparation

`data/` 디렉토리의 Stage 1 (World Modeling) 데이터를 Train/Test Split 후 LLaMA-Factory에 등록합니다.

**Task**: UI State (XML) + Action → Next UI State (XML)

**데이터 구조:**
- **Format**: ShareGPT (multimodal)

| File | MobiBench | AndroidControl |
|------|-----------|----------------|
| gui-model_stage1.jsonl | 3,145건 | 71,047건 |
| gui-model_stage1_train.jsonl | ~2,987건 (95%) | ~67,494건 (95%) |
| gui-model_stage1_test.jsonl | ~158건 (5%) | ~3,553건 (5%) |
| Images | 3,655개 PNG | (jsonl 내 참조) |

**Split 전략:**
- Random Split (seed=42, 재현 가능)
- 비율: 0.95 : 0.05


In [ ]:
import json
from pathlib import Path

LF_PATH = Path(LF_ROOT)
LF_DATA_DIR = LF_PATH / "data"

SHAREGPT_TAGS = {
    "role_tag": "from",
    "content_tag": "value",
    "user_tag": "human",
    "assistant_tag": "gpt",
    "system_tag": "system"
}

dataset_info_path = LF_DATA_DIR / "dataset_info.json"
if dataset_info_path.exists():
    with open(dataset_info_path, 'r', encoding='utf-8') as f:
        dataset_info = json.load(f)
else:
    dataset_info = {}

# Dataset registration is model-independent; iterate _DATASET_CONFIG directly
_any_model = next(iter(CONFIGS))
for ds_name, cfg in CONFIGS[_any_model].items():
    DATA_PATH = Path(cfg["data_dir"])
    print(f"\n{'='*60}")
    print(f"=== {ds_name} Stage 1 Data Registration ===\n")

    for fname in ["gui-model_stage1_train.jsonl", "gui-model_stage1_test.jsonl"]:
        fpath = DATA_PATH / fname
        if not fpath.exists():
            raise FileNotFoundError(
                f"{fpath} not found. Run split first:\n"
                f"  python scripts/split_data.py --dataset {ds_name}"
            )
        with open(fpath, 'r') as f:
            count = sum(1 for _ in f)
        print(f"[OK] {fname} ({count} entries, {fpath.stat().st_size / 1024 / 1024:.1f} MB)")

    DATASETS_TO_REGISTER = {
        cfg["ds_s1_train"]: {
            "file_name": f"../../data/{ds_name}/gui-model_stage1_train.jsonl",
            "formatting": "sharegpt",
            "columns": {"messages": "messages", "images": "images"},
            "tags": SHAREGPT_TAGS
        },
        cfg["ds_s1_test"]: {
            "file_name": f"../../data/{ds_name}/gui-model_stage1_test.jsonl",
            "formatting": "sharegpt",
            "columns": {"messages": "messages", "images": "images"},
            "tags": SHAREGPT_TAGS
        }
    }

    for name, config in DATASETS_TO_REGISTER.items():
        dataset_info[name] = config
        print(f"[Registered] {name} -> {config['file_name']}")

with open(dataset_info_path, 'w', encoding='utf-8') as f:
    json.dump(dataset_info, f, ensure_ascii=False, indent=2)

print(f"\n=== dataset_info.json saved ({len(dataset_info)} datasets) ===")

In [ ]:
import json
from pathlib import Path

_any_model = next(iter(CONFIGS))
for ds_name, cfg in CONFIGS[_any_model].items():
    DATA_PATH = Path(cfg["data_dir"])
    print(f"\n{'='*60}")
    print(f"=== {ds_name} Stage 1 Dataset Statistics ===\n")

    for fname in ["gui-model_stage1.jsonl", "gui-model_stage1_train.jsonl", "gui-model_stage1_test.jsonl"]:
        fpath = DATA_PATH / fname
        if fpath.exists():
            with open(fpath, 'r') as f:
                entries = [json.loads(line) for line in f if line.strip()]
            sample = entries[0] if entries else None
            msg_count = len(sample['messages']) if sample else 0
            img_count = len(sample.get('images', [])) if sample else 0
            print(f"  {fname}")
            print(f"   Entries: {len(entries)}")
            print(f"   Messages per sample: {msg_count}")
            print(f"   Images per sample: {img_count}")
            print(f"   File size: {fpath.stat().st_size / 1024 / 1024:.1f} MB")
            print()

    img_dir = DATA_PATH / "images"
    if img_dir.exists():
        imgs = list(img_dir.glob("*.png"))
        total_size = sum(f.stat().st_size for f in imgs) / 1024 / 1024 / 1024
        print(f"  Image directory: {img_dir}")
        print(f"   Total images: {len(imgs)}")
        print(f"   Total size: {total_size:.2f} GB")

## 2. Stage 2 Data Preparation

Stage 2 (Action Prediction) 데이터를 **Train/Test Split** 후 LLaMA-Factory에 등록합니다.

**Task**: Screenshot + UI State (XML) + Task Description → Action (JSON)

**데이터 구조:**
- **Format**: ShareGPT (multimodal)
- **Images**: Stage 1과 동일 (공유)

| File | MobiBench | AndroidControl |
|------|-----------|----------------|
| gui-model_stage2.jsonl | 3,147건 | 91,677건 |
| gui-model_stage2_train.jsonl | ~2,987건 (95%) | ~87,090건 (95%) |
| gui-model_stage2_test.jsonl | ~160건 (5%) | ~4,587건 (5%) |

**Split 전략:**
- Stratified Split (action type 비율 유지)
- Random seed: 42 (재현 가능)


In [ ]:
import json
from pathlib import Path

LF_PATH = Path(LF_ROOT)
LF_DATA_DIR = LF_PATH / "data"

SHAREGPT_TAGS = {
    "role_tag": "from",
    "content_tag": "value",
    "user_tag": "human",
    "assistant_tag": "gpt",
    "system_tag": "system"
}

dataset_info_path = LF_DATA_DIR / "dataset_info.json"
if dataset_info_path.exists():
    with open(dataset_info_path, 'r', encoding='utf-8') as f:
        dataset_info = json.load(f)
else:
    dataset_info = {}

_any_model = next(iter(CONFIGS))
for ds_name, cfg in CONFIGS[_any_model].items():
    DATA_PATH = Path(cfg["data_dir"])
    print(f"\n{'='*60}")
    print(f"=== {ds_name} Stage 2 Data Registration ===\n")

    for fname in ["gui-model_stage2_train.jsonl", "gui-model_stage2_test.jsonl"]:
        fpath = DATA_PATH / fname
        if not fpath.exists():
            raise FileNotFoundError(
                f"{fpath} not found. Run split first:\n"
                f"  python scripts/split_data.py --dataset {ds_name}"
            )
        with open(fpath, 'r') as f:
            count = sum(1 for _ in f)
        print(f"[OK] {fname} ({count} entries, {fpath.stat().st_size / 1024 / 1024:.1f} MB)")

    DATASETS_TO_REGISTER = {
        cfg["ds_s2_train"]: {
            "file_name": f"../../data/{ds_name}/gui-model_stage2_train.jsonl",
            "formatting": "sharegpt",
            "columns": {"messages": "messages", "images": "images"},
            "tags": SHAREGPT_TAGS
        },
        cfg["ds_s2_test"]: {
            "file_name": f"../../data/{ds_name}/gui-model_stage2_test.jsonl",
            "formatting": "sharegpt",
            "columns": {"messages": "messages", "images": "images"},
            "tags": SHAREGPT_TAGS
        }
    }

    for name, config in DATASETS_TO_REGISTER.items():
        dataset_info[name] = config
        print(f"[Registered] {name} -> {config['file_name']}")

with open(dataset_info_path, 'w', encoding='utf-8') as f:
    json.dump(dataset_info, f, ensure_ascii=False, indent=2)

print(f"\n=== dataset_info.json saved ({len(dataset_info)} datasets) ===")

In [ ]:
import json
from pathlib import Path
from collections import Counter

_any_model = next(iter(CONFIGS))
for ds_name, cfg in CONFIGS[_any_model].items():
    DATA_PATH = Path(cfg["data_dir"])
    print(f"\n{'='*60}")
    print(f"=== {ds_name} Stage 2 Dataset Statistics ===\n")

    for fname in ["gui-model_stage2.jsonl", "gui-model_stage2_train.jsonl", "gui-model_stage2_test.jsonl"]:
        fpath = DATA_PATH / fname
        if fpath.exists():
            with open(fpath, 'r') as f:
                entries = [json.loads(line) for line in f if line.strip()]
            print(f"  {fname}")
            print(f"   Entries: {len(entries)}")
            print(f"   File size: {fpath.stat().st_size / 1024 / 1024:.1f} MB")

            action_types = []
            for entry in entries:
                try:
                    action = json.loads(entry['messages'][-1]['value'])
                    action_types.append(action.get('type', 'unknown'))
                except:
                    action_types.append('parse_error')

            type_counts = Counter(action_types)
            total = len(action_types)
            print(f"   Action type distribution:")
            for atype, count in type_counts.most_common():
                print(f"     {atype}: {count} ({count/total:.1%})")
            print()

In [ ]:
import json
from pathlib import Path
from collections import Counter

_any_model = next(iter(CONFIGS))
for ds_name, cfg in CONFIGS[_any_model].items():
    DATA_PATH = Path(cfg["data_dir"])
    print(f"\n{'='*60}")
    print(f"=== {ds_name} Stage 2 Action Type Analysis ===")

    train_path = DATA_PATH / "gui-model_stage2_train.jsonl"
    test_path = DATA_PATH / "gui-model_stage2_test.jsonl"

    for label, fpath in [("Train", train_path), ("Test", test_path)]:
        if not fpath.exists():
            print(f"[SKIP] {label}: {fpath.name} not found")
            continue

        with open(fpath, 'r') as f:
            entries = [json.loads(line) for line in f if line.strip()]

        action_types = []
        for entry in entries:
            try:
                action = json.loads(entry['messages'][-1]['value'])
                action_types.append(action.get('type', 'unknown'))
            except:
                action_types.append('parse_error')

        type_counts = Counter(action_types)
        total = len(action_types)
        print(f"\n  {label} Set Action Types ({total} entries)")
        for atype, count in type_counts.most_common():
            print(f"    {atype}: {count} ({count/total:.1%})")

## 3. Stage 1 SFT Training (default: full)

노트북 실행 셀은 **full mode** 산출물 경로를 기준으로 작성되어 있다. LoRA mode 학습은
`bash scripts/stage1_train.sh --stage1-mode lora` 로 실행한다.


In [ ]:
os.chdir(LF_ROOT)

## [Qwen2-VL-2B] [MobiBench] Stage 1 Full Fine-tuning
!DISABLE_VERSION_CHECK=1 FORCE_TORCHRUN=1 NNODES=1 NPROC_PER_NODE=${NPROC_PER_NODE:-2} \
    llamafactory-cli train custom/GUI-Model-MB/stage1_full/qwen2-vl-2b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2-VL-2B] [AndroidControl] Stage 1 Full Fine-tuning
!DISABLE_VERSION_CHECK=1 FORCE_TORCHRUN=1 NNODES=1 NPROC_PER_NODE=${NPROC_PER_NODE:-2} \
    llamafactory-cli train custom/GUI-Model-AC/stage1_full/qwen2-vl-2b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2-VL-7B] [MobiBench] Stage 1 Full Fine-tuning
!DISABLE_VERSION_CHECK=1 FORCE_TORCHRUN=1 NNODES=1 NPROC_PER_NODE=${NPROC_PER_NODE:-2} \
    llamafactory-cli train custom/GUI-Model-MB/stage1_full/qwen2-vl-7b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2-VL-7B] [AndroidControl] Stage 1 Full Fine-tuning
!DISABLE_VERSION_CHECK=1 FORCE_TORCHRUN=1 NNODES=1 NPROC_PER_NODE=${NPROC_PER_NODE:-2} \
    llamafactory-cli train custom/GUI-Model-AC/stage1_full/qwen2-vl-7b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2.5-VL-3B] [MobiBench] Stage 1 Full Fine-tuning
!DISABLE_VERSION_CHECK=1 FORCE_TORCHRUN=1 NNODES=1 NPROC_PER_NODE=${NPROC_PER_NODE:-2} \
    llamafactory-cli train custom/GUI-Model-MB/stage1_full/qwen2.5-vl-3b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2.5-VL-3B] [AndroidControl] Stage 1 Full Fine-tuning
!DISABLE_VERSION_CHECK=1 FORCE_TORCHRUN=1 NNODES=1 NPROC_PER_NODE=${NPROC_PER_NODE:-2} \
    llamafactory-cli train custom/GUI-Model-AC/stage1_full/qwen2.5-vl-3b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2.5-VL-7B] [MobiBench] Stage 1 Full Fine-tuning
!DISABLE_VERSION_CHECK=1 FORCE_TORCHRUN=1 NNODES=1 NPROC_PER_NODE=${NPROC_PER_NODE:-2} \
    llamafactory-cli train custom/GUI-Model-MB/stage1_full/qwen2.5-vl-7b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2.5-VL-7B] [AndroidControl] Stage 1 Full Fine-tuning
!DISABLE_VERSION_CHECK=1 FORCE_TORCHRUN=1 NNODES=1 NPROC_PER_NODE=${NPROC_PER_NODE:-2} \
    llamafactory-cli train custom/GUI-Model-AC/stage1_full/qwen2.5-vl-7b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen3-VL-2B] [MobiBench] Stage 1 Full Fine-tuning
!DISABLE_VERSION_CHECK=1 FORCE_TORCHRUN=1 NNODES=1 NPROC_PER_NODE=${NPROC_PER_NODE:-2} \
    llamafactory-cli train custom/GUI-Model-MB/stage1_full/qwen3-vl-2b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen3-VL-2B] [AndroidControl] Stage 1 Full Fine-tuning
!DISABLE_VERSION_CHECK=1 FORCE_TORCHRUN=1 NNODES=1 NPROC_PER_NODE=${NPROC_PER_NODE:-2} \
    llamafactory-cli train custom/GUI-Model-AC/stage1_full/qwen3-vl-2b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen3-VL-4B] [MobiBench] Stage 1 Full Fine-tuning
!DISABLE_VERSION_CHECK=1 FORCE_TORCHRUN=1 NNODES=1 NPROC_PER_NODE=${NPROC_PER_NODE:-2} \
    llamafactory-cli train custom/GUI-Model-MB/stage1_full/qwen3-vl-4b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen3-VL-4B] [AndroidControl] Stage 1 Full Fine-tuning
!DISABLE_VERSION_CHECK=1 FORCE_TORCHRUN=1 NNODES=1 NPROC_PER_NODE=${NPROC_PER_NODE:-2} \
    llamafactory-cli train custom/GUI-Model-AC/stage1_full/qwen3-vl-4b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen3-VL-8B] [MobiBench] Stage 1 Full Fine-tuning
!DISABLE_VERSION_CHECK=1 FORCE_TORCHRUN=1 NNODES=1 NPROC_PER_NODE=${NPROC_PER_NODE:-2} \
    llamafactory-cli train custom/GUI-Model-MB/stage1_full/qwen3-vl-8b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen3-VL-8B] [AndroidControl] Stage 1 Full Fine-tuning
!DISABLE_VERSION_CHECK=1 FORCE_TORCHRUN=1 NNODES=1 NPROC_PER_NODE=${NPROC_PER_NODE:-2} \
    llamafactory-cli train custom/GUI-Model-AC/stage1_full/qwen3-vl-8b_world-model.yaml

In [ ]:
os.chdir(BASE_DIR)

## [Gemma-4-E2B] [MobiBench] Stage 1 Full Fine-tuning | Unsloth
!bash "$BASE_DIR/scripts/stage1_train.sh" --model gemma-4-e2b --dataset MB


In [ ]:
os.chdir(BASE_DIR)

## [Gemma-4-E2B] [AndroidControl] Stage 1 Full Fine-tuning | Unsloth
!bash "$BASE_DIR/scripts/stage1_train.sh" --model gemma-4-e2b --dataset AC


In [ ]:
os.chdir(BASE_DIR)

## [Gemma-4-E4B] [MobiBench] Stage 1 Full Fine-tuning | Unsloth
!bash "$BASE_DIR/scripts/stage1_train.sh" --model gemma-4-e4b --dataset MB


In [ ]:
os.chdir(BASE_DIR)

## [Gemma-4-E4B] [AndroidControl] Stage 1 Full Fine-tuning | Unsloth
!bash "$BASE_DIR/scripts/stage1_train.sh" --model gemma-4-e4b --dataset AC


In [ ]:
os.chdir(LF_ROOT)

## [LLaVA-v1.6-Mistral-7B] [MobiBench] Stage 1 Full Fine-tuning
!DISABLE_VERSION_CHECK=1 FORCE_TORCHRUN=1 NNODES=1 NPROC_PER_NODE=${NPROC_PER_NODE:-2} \
    llamafactory-cli train custom/GUI-Model-MB/stage1_full/llava-v1.6-mistral-7b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [LLaVA-v1.6-Mistral-7B] [AndroidControl] Stage 1 Full Fine-tuning
!DISABLE_VERSION_CHECK=1 FORCE_TORCHRUN=1 NNODES=1 NPROC_PER_NODE=${NPROC_PER_NODE:-2} \
    llamafactory-cli train custom/GUI-Model-AC/stage1_full/llava-v1.6-mistral-7b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [LLaVA-v1.6-Vicuna-7B] [MobiBench] Stage 1 Full Fine-tuning
!DISABLE_VERSION_CHECK=1 FORCE_TORCHRUN=1 NNODES=1 NPROC_PER_NODE=${NPROC_PER_NODE:-2} \
    llamafactory-cli train custom/GUI-Model-MB/stage1_full/llava-v1.6-vicuna-7b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [LLaVA-v1.6-Vicuna-7B] [AndroidControl] Stage 1 Full Fine-tuning
!DISABLE_VERSION_CHECK=1 FORCE_TORCHRUN=1 NNODES=1 NPROC_PER_NODE=${NPROC_PER_NODE:-2} \
    llamafactory-cli train custom/GUI-Model-AC/stage1_full/llava-v1.6-vicuna-7b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [LLaMA3-LLaVA-Next-8B] [MobiBench] Stage 1 Full Fine-tuning
!DISABLE_VERSION_CHECK=1 FORCE_TORCHRUN=1 NNODES=1 NPROC_PER_NODE=${NPROC_PER_NODE:-2} \
    llamafactory-cli train custom/GUI-Model-MB/stage1_full/llama3-llava-next-8b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [LLaMA3-LLaVA-Next-8B] [AndroidControl] Stage 1 Full Fine-tuning
!DISABLE_VERSION_CHECK=1 FORCE_TORCHRUN=1 NNODES=1 NPROC_PER_NODE=${NPROC_PER_NODE:-2} \
    llamafactory-cli train custom/GUI-Model-AC/stage1_full/llama3-llava-next-8b_world-model.yaml

## 4. Stage 1 Merge & Upload to HuggingFace (전 epoch push)

> **실행 전 필수**: Section 4 의 Stage 1 평가를 완료한 뒤 아래 **Stage 1 Merge YAML** 셀을 재실행하여 `BEST_CHECKPOINT` 가 반영된 Merge YAML 로 덮어쓰세요. 그 후 아래 export cell 을 실행합니다. Merge 결과물은 `outputs/{MODEL}/{DS}/stage1_merged/` 에 저장됩니다.


학습된 Stage 1 체크포인트 중 **Stage 1 평가 단계에서 자동 선택된 Hungarian F1 winner** 를 병합(merge)하고 HuggingFace Hub에 업로드합니다.

**선택 방식:**
1. Stage 1 학습 결과 `saves/{MODEL}/{DS}/stage1_full/full_world_model/checkpoint-*/` 다수 생성
2. Stage 1 평가 파이프라인이 각 체크포인트별 Hungarian 메트릭 계산 → `BEST_CHECKPOINT` 파일 기록
3. 아래 **Stage 1 Merge YAML** 셀이 그 파일을 읽어 merge YAML 의 `model_name_or_path` 를 winner 체크포인트로 설정
4. `BEST_CHECKPOINT` 없는 모델/데이터셋 조합은 `[SKIP]` 경고와 함께 건너뜀 — stage1 평가 완료 후 Merge YAML 셀 재실행

**업로드 대상 (12개 모델 × 2개 데이터셋):**

| Dataset | HuggingFace Model ID 패턴 | 예시 (Qwen3-VL-8B) |
|---------|--------------------------|---------------------|
| MobiBench | `SaFD-00/{model}-mb-stage1-world-model` | `SaFD-00/qwen3-vl-8b-mb-stage1-world-model` |
| AndroidControl | `SaFD-00/{model}-ac-stage1-world-model` | `SaFD-00/qwen3-vl-8b-ac-stage1-world-model` |

> `{model}` = 모델 short_name (예: `qwen2-vl-2b`, `qwen3-vl-8b`, `gemma-4-e2b`, `llama3-llava-next-8b` 등)

> **주의:** Merge 시 quantized 모델이나 quantization_bit를 사용하지 마세요. 아래 **Stage 1 Merge YAML** 셀은 **Stage 1 평가(winner 선택)** 이후에 재실행해야 올바른 YAML 이 생성됩니다.

> **흐름**: `stage1_train.sh` → `stage1_merge.sh` (모든 epoch checkpoint 를 각각 merge + HF push) → `stage1_eval.sh` (HF pull 로 sweep + winner 선정). HF repo id 규칙은 `_common.sh::hf_repo_id_stage1` 단일 정의.


#### [LlamaFactory] Stage 1 Merge YAML

`LlamaFactory/examples/custom/GUI-Model-{MB,AC}/stage1_merge/{MODEL}_world-model.yaml` 생성. `outputs/{DS}/adapters/{MODEL}_stage1_full/BEST_CHECKPOINT` 파일을 읽어 Hungarian F1 winner 체크포인트를 `model_name_or_path` 에 주입. 파일이 없으면 해당 조합을 `[SKIP]` 경고 후 건너뛰며, 평가 완료 후 재실행하면 건너뛴 항목도 생성. Merge 결과물은 `outputs/{DS}/merged/{MODEL}_stage1_full/` 에 저장. `backend == "llamafactory"` 모델만 대상 (qwen / llava 계열).


In [ ]:
from pathlib import Path

_created, _skipped = 0, 0

# LF Stage 1 Merge YAML — full / lora 모드별로 생성.
# LoRA 모드는 finetuning_type: lora + adapter_name_or_path 블록을 추가,
# model_name_or_path 는 원본 base model (cfg["model_id"]) 을 참조.
for MODE in ["full", "lora"]:
    for model_key, ds_configs in CONFIGS.items():
        for ds_name, cfg in ds_configs.items():
            if cfg["model_config"].get("backend") == "unsloth":
                continue
            lf_sub = cfg["lf_subfolder"]
            yaml_dir = Path(LF_ROOT) / "examples" / "custom" / lf_sub / f"stage1_merge_{MODE}"
            yaml_dir.mkdir(parents=True, exist_ok=True)

            save_path_rel = cfg[f"save_s1_{MODE}"]
            train_dir = Path(BASE_DIR) / save_path_rel.lstrip("../")
            best_file = train_dir / "BEST_CHECKPOINT"
            if not best_file.exists():
                print(f"[SKIP] [{model_key}/{ds_name}/{MODE}] BEST_CHECKPOINT not found at {best_file}. "
                      f"Run stage1 evaluation first.")
                _skipped += 1
                continue
            ckpt_name = best_file.read_text(encoding="utf-8").strip()
            ckpt_path = f"{save_path_rel}/{ckpt_name}"
            selection = f"Hungarian F1 winner: {ckpt_name}"

            if MODE == "full":
                model_block = f"""\
### model
model_name_or_path: {ckpt_path}
trust_remote_code: true
template: {cfg["template"]}
"""
            else:
                model_block = f"""\
### model
model_name_or_path: {cfg["model_id"]}
adapter_name_or_path: {ckpt_path}
trust_remote_code: true
finetuning_type: lora
template: {cfg["template"]}
"""

            MERGE_STAGE1_CONFIG = f"""\
{model_block}
### export
export_dir: {cfg[f"out_s1_merged_{MODE}"]}
export_size: 5
export_device: cpu
export_legacy_format: false
export_hub_model_id: {cfg[f"hf_s1_model_{MODE}"]}
"""

            fpath = yaml_dir / f"{model_key}_world-model.yaml"
            with open(fpath, 'w') as f:
                f.write(MERGE_STAGE1_CONFIG)
            print(f"[Created/{MODE}] {fpath.relative_to(Path(LF_ROOT))}")
            print(f"  selection: {selection}")
            print(f"  ckpt:      {ckpt_path}")
            print(f"  export:    {cfg[f'out_s1_merged_{MODE}']}")
            print(f"  hub_id:    {cfg[f'hf_s1_model_{MODE}']}")
            print()
            _created += 1

print(f"--- Stage 1 Merge YAML: {_created} created, {_skipped} skipped ---")
if _skipped > 0:
    print("Re-run this cell after completing stage1 evaluation for the skipped models.")


In [ ]:
os.chdir(LF_ROOT)

## [Qwen2-VL-2B] [MobiBench] Stage 1 Merge & HuggingFace Hub Upload
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-MB/stage1_merge/qwen2-vl-2b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2-VL-2B] [AndroidControl] Stage 1 Merge & HuggingFace Hub Upload
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-AC/stage1_merge/qwen2-vl-2b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2-VL-7B] [MobiBench] Stage 1 Merge & HuggingFace Hub Upload
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-MB/stage1_merge/qwen2-vl-7b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2-VL-7B] [AndroidControl] Stage 1 Merge & HuggingFace Hub Upload
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-AC/stage1_merge/qwen2-vl-7b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2.5-VL-3B] [MobiBench] Stage 1 Merge & HuggingFace Hub Upload
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-MB/stage1_merge/qwen2.5-vl-3b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2.5-VL-3B] [AndroidControl] Stage 1 Merge & HuggingFace Hub Upload
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-AC/stage1_merge/qwen2.5-vl-3b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2.5-VL-7B] [MobiBench] Stage 1 Merge & HuggingFace Hub Upload
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-MB/stage1_merge/qwen2.5-vl-7b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2.5-VL-7B] [AndroidControl] Stage 1 Merge & HuggingFace Hub Upload
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-AC/stage1_merge/qwen2.5-vl-7b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen3-VL-2B] [MobiBench] Stage 1 Merge & HuggingFace Hub Upload
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-MB/stage1_merge/qwen3-vl-2b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen3-VL-2B] [AndroidControl] Stage 1 Merge & HuggingFace Hub Upload
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-AC/stage1_merge/qwen3-vl-2b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen3-VL-4B] [MobiBench] Stage 1 Merge & HuggingFace Hub Upload
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-MB/stage1_merge/qwen3-vl-4b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen3-VL-4B] [AndroidControl] Stage 1 Merge & HuggingFace Hub Upload
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-AC/stage1_merge/qwen3-vl-4b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen3-VL-8B] [MobiBench] Stage 1 Merge & HuggingFace Hub Upload
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-MB/stage1_merge/qwen3-vl-8b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen3-VL-8B] [AndroidControl] Stage 1 Merge & HuggingFace Hub Upload
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-AC/stage1_merge/qwen3-vl-8b_world-model.yaml

In [ ]:
os.chdir(BASE_DIR)

## [Gemma-4-E2B] [MobiBench] Stage 1 Merge & HuggingFace Hub Upload | Unsloth
!bash "$BASE_DIR/scripts/stage1_merge.sh" --model gemma-4-e2b --dataset MB


In [ ]:
os.chdir(BASE_DIR)

## [Gemma-4-E2B] [AndroidControl] Stage 1 Merge & HuggingFace Hub Upload | Unsloth
!bash "$BASE_DIR/scripts/stage1_merge.sh" --model gemma-4-e2b --dataset AC


In [ ]:
os.chdir(BASE_DIR)

## [Gemma-4-E4B] [MobiBench] Stage 1 Merge & HuggingFace Hub Upload | Unsloth
!bash "$BASE_DIR/scripts/stage1_merge.sh" --model gemma-4-e4b --dataset MB


In [ ]:
os.chdir(BASE_DIR)

## [Gemma-4-E4B] [AndroidControl] Stage 1 Merge & HuggingFace Hub Upload | Unsloth
!bash "$BASE_DIR/scripts/stage1_merge.sh" --model gemma-4-e4b --dataset AC


In [ ]:
os.chdir(LF_ROOT)

## [LLaVA-v1.6-Mistral-7B] [MobiBench] Stage 1 Merge & HuggingFace Hub Upload
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-MB/stage1_merge/llava-v1.6-mistral-7b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [LLaVA-v1.6-Mistral-7B] [AndroidControl] Stage 1 Merge & HuggingFace Hub Upload
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-AC/stage1_merge/llava-v1.6-mistral-7b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [LLaVA-v1.6-Vicuna-7B] [MobiBench] Stage 1 Merge & HuggingFace Hub Upload
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-MB/stage1_merge/llava-v1.6-vicuna-7b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [LLaVA-v1.6-Vicuna-7B] [AndroidControl] Stage 1 Merge & HuggingFace Hub Upload
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-AC/stage1_merge/llava-v1.6-vicuna-7b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [LLaMA3-LLaVA-Next-8B] [MobiBench] Stage 1 Merge & HuggingFace Hub Upload
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-MB/stage1_merge/llama3-llava-next-8b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [LLaMA3-LLaVA-Next-8B] [AndroidControl] Stage 1 Merge & HuggingFace Hub Upload
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-AC/stage1_merge/llama3-llava-next-8b_world-model.yaml

### 5. Stage 1 Evaluation Pipeline

**흐름: train → eval → merge.** eval 은 HF Hub 의존 없이 로컬 per-epoch checkpoint 만 소비한다.

1. **Baseline (zero-shot)** — `{model_id}` 에 대해 `vllm_infer.py` 로 `outputs/{DS}/eval/{MODEL}/stage1_eval/base/` 생성.
2. **Checkpoint sweep** — `outputs/{DS}/adapters/{MODEL}_stage1_${MODE}/epoch-*` 을 순회.
   - `full`: `--model_name_or_path <checkpoint_dir>`
   - `lora`: `--model_name_or_path {model_id} --adapter_name_or_path <checkpoint_dir>` + `max_lora_rank=8`
   - 결과: `outputs/{DS}/eval/{MODEL}/stage1_eval/{full|lora}_world_model/checkpoint-N/(generated_predictions.jsonl|hungarian_metrics.json)`
3. **Winner 선정** — `_hungarian_eval.py select` 가 `avg_hungarian_f1` 최댓값 checkpoint 를 `outputs/{DS}/adapters/{MODEL}_stage1_${MODE}/BEST_CHECKPOINT[.json]` 에 기록. 이어지는 `stage1_merge.sh` 가 이 파일을 소비한다.

쉘 경로는 `bash scripts/stage1_eval.sh --model {MODEL} --dataset {DS} --stage1-mode {full|lora}`.

> **HF Hub sweep**: `adapters/.../checkpoint-*/` 에서 epoch 을 추출해 `SaFD-00/{short}-{slug}stage1-{mode}-world-model-epoch{E}` 로 HF repo 를 조립 → `vllm_infer.py --model_name_or_path <HF id>`. 쉘 경로: `bash scripts/stage1_eval.sh --model {MODEL} --dataset {DS} --stage1-mode {full|lora}`.


In [ ]:
import json, math, os
from pathlib import Path
from datetime import datetime
import matplotlib.pyplot as plt

# Stage 1 Eval-Loss report — baseline vs per-checkpoint world-model sweep.
# winner 는 BEST_CHECKPOINT (adapter 경로) 가 있으면 우선 참조; 없으면 step 기준 최신.

def _load_eval_loss(p: Path):
    if not p.exists():
        return None
    try:
        with open(p, "r") as f:
            return json.load(f).get("eval_loss")
    except Exception:
        return None

for model_key, ds_configs in CONFIGS.items():
    for ds_name, cfg in ds_configs.items():
        if cfg["model_config"].get("backend") == "unsloth":
            continue
        short_name = cfg["short_name"]
        ds_code    = cfg["output_prefix"].rstrip("/")
        eval_loss_dir = Path(BASE_DIR) / "outputs" / ds_code / "eval" / short_name / "stage1_eval" / "eval_loss"
        base_loss = _load_eval_loss(eval_loss_dir / "base" / "eval_results.json")
        if base_loss is None:
            print(f"[SKIP] {model_key}/{ds_name}: baseline eval_results.json 없음")
            continue

        # 두 모드 각각 per-checkpoint 집계
        mode_rows = {}
        for MODE in ("full", "lora"):
            wm_dir = eval_loss_dir / f"{MODE}_world_model"
            per_ckpt = []
            if wm_dir.is_dir():
                for cp in sorted(wm_dir.glob("epoch-*"),
                                 key=lambda p: int(p.name.split("-")[-1]) if p.name.split("-")[-1].isdigit() else -1):
                    l = _load_eval_loss(cp / "eval_results.json")
                    if l is not None:
                        per_ckpt.append((cp.name, l))
            if not per_ckpt:
                continue

            # winner: BEST_CHECKPOINT 참고 (adapter 디렉토리)
            best_file = Path(BASE_DIR) / "outputs" / ds_code / "adapters" / f"{short_name}_stage1_{MODE}" / "BEST_CHECKPOINT"
            winner_name = best_file.read_text().strip() if best_file.exists() else per_ckpt[-1][0]
            mode_rows[MODE] = {"checkpoints": per_ckpt, "winner": winner_name}

        if not mode_rows:
            print(f"[SKIP] {model_key}/{ds_name}: no world-model checkpoint eval_results 발견")
            continue

        base_ppl = math.exp(base_loss)
        print(f"\n=== {model_key} / {ds_name} Stage 1 Eval-Loss ===")
        print(f"Baseline (Zero-shot)  eval_loss={base_loss:.4f}  ppl={base_ppl:.4f}")
        for MODE, info in mode_rows.items():
            print(f"[{MODE}_world_model]")
            for name, l in info["checkpoints"]:
                mark = " <-- winner" if name == info["winner"] else ""
                print(f"  {name:<18} eval_loss={l:.4f}  ppl={math.exp(l):.4f}{mark}")

        # Winner checkpoint 를 대표로 써서 막대그래프 — baseline vs winner (모드별)
        variants = [("Base\n(Zero-shot)", base_loss)]
        for MODE, info in mode_rows.items():
            w_loss = dict(info["checkpoints"])[info["winner"]]
            variants.append((f"{MODE}\n({info['winner']})", w_loss))
        labels, losses = zip(*variants)
        colors = ["#9E9E9E"] + ["#FF5722", "#3F51B5"][:len(variants) - 1]
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        b0 = axes[0].bar(labels, losses, color=colors, width=0.5)
        axes[0].set_title("Eval Loss")
        axes[0].set_ylabel("Cross-Entropy Loss")
        for bar, val in zip(b0, losses):
            axes[0].text(bar.get_x()+bar.get_width()/2, val+max(losses)*0.02, f"{val:.4f}", ha="center", va="bottom", fontsize=11, fontweight="bold")
        axes[0].set_ylim(0, max(losses) * 1.3); axes[0].grid(axis="y", alpha=0.3)
        perps = [math.exp(l) for l in losses]
        b1 = axes[1].bar(labels, perps, color=colors, width=0.5)
        axes[1].set_title("Perplexity"); axes[1].set_ylabel("Perplexity = exp(loss)")
        for bar, val in zip(b1, perps):
            axes[1].text(bar.get_x()+bar.get_width()/2, val+max(perps)*0.02, f"{val:.4f}", ha="center", va="bottom", fontsize=11, fontweight="bold")
        axes[1].set_ylim(0, max(perps) * 1.3); axes[1].grid(axis="y", alpha=0.3)
        plt.suptitle(f"Stage 1 Eval-Loss ({model_key} / {ds_name}) — baseline vs winner", fontsize=13, fontweight="bold")
        plt.tight_layout()
        img_path = eval_loss_dir / "stage1_eval_loss_evaluation.png"
        plt.savefig(img_path, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"[Saved] {img_path}")

        # Markdown report
        now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        rlines = [f"# Stage 1 Eval-Loss Report: Base vs per-checkpoint sweep ({model_key} / {ds_name})",
                  f"\n> Generated: {now}\n",
                  "## Experiment Setup\n",
                  "| Item | Value |",
                  "|------|-------|",
                  f"| Model | {cfg['model_id']} |",
                  f"| Dataset | {ds_name} |",
                  f"| Test Dataset | {cfg['ds_s1_test']} |",
                  ""]
        rlines.append("## Baseline\n")
        rlines += ["| Metric | Base (Zero-shot) |",
                   "|--------|------------------|",
                   f"| Eval Loss | {base_loss:.4f} |",
                   f"| Perplexity | {base_ppl:.4f} |", ""]
        for MODE, info in mode_rows.items():
            rlines.append(f"## World-Model (stage1_{MODE}) — per-checkpoint\n")
            rlines += [f"| Checkpoint | Eval Loss | Perplexity | vs Base |",
                       f"|------------|-----------|------------|---------|"]
            for name, l in info["checkpoints"]:
                mark = "  **(winner)**" if name == info["winner"] else ""
                rlines.append(f"| {name}{mark} | {l:.4f} | {math.exp(l):.4f} | {(base_loss - l)/base_loss:+.1%} |")
            rlines.append("")
        report = "\n".join(rlines)
        report_path = eval_loss_dir / "evaluation_report.md"
        report_path.parent.mkdir(parents=True, exist_ok=True)
        report_path.write_text(report, encoding="utf-8")
        print(f"[Saved] {report_path}")


In [ ]:
os.chdir(LF_ROOT)

for model_key, ds_configs in CONFIGS.items():
    for ds_name, cfg in ds_configs.items():
        short_name = cfg["short_name"]
        _template = cfg["template"]
        _enable_thinking = "--enable_thinking False" if _template.startswith("qwen3_vl") else ""

        ## Hungarian Matching: Baseline (Zero-shot)
        _out = f"../outputs/{cfg['output_prefix'].rstrip('/')}/eval/{short_name}/stage1_eval/base"
        !mkdir -p {_out}
        !python scripts/vllm_infer.py \
            --model_name_or_path {cfg["model_id"]} \
            --dataset {cfg['ds_s1_test']} \
            --template {_template} \
            --cutoff_len 8192 \
            --image_max_pixels {cfg["model_config"]["image_max_pixels"]} \
            {_enable_thinking} \
            --save_name {_out}/generated_predictions.jsonl \
            --matrix_save_name {_out}/predict_results.json

In [ ]:
## Stage 1 Hungarian Matching — HF Hub epoch 별 merged 모델 sweep + winner 선정
## epoch 리스트를 명시적으로 지정 (shell `stage1_eval.sh --epochs` 와 동일한 전환).
## 로컬 adapter/merged 디렉토리는 조회하지 않고 HF Hub merged repo 만 pull 한다.
from pathlib import Path
os.chdir(LF_ROOT)

EPOCHS = [1, 2, 3]
HF_SLUG = {"MobiBench": "mb-", "AndroidControl": "ac-"}

for model_key, ds_configs in CONFIGS.items():
    for _ds_name, _cfg in ds_configs.items():
        if _cfg["model_config"].get("backend") == "unsloth":
            continue
        short_name = _cfg["short_name"]
        ds_code    = _cfg["output_prefix"].rstrip("/")
        _slug      = HF_SLUG[_ds_name]
        _template  = _cfg["template"]
        _enable_thinking = "--enable_thinking False" if _template.startswith("qwen3_vl") else ""
        _img_px    = _cfg["model_config"]["image_max_pixels"]
        _test_jsonl = f"{BASE_DIR}/data/{_ds_name}/gui-model_stage1_test.jsonl"
        _vllm_cfg  = '{"gpu_memory_utilization": 0.80}'   # merged 단일 모델 — max_lora_rank 불필요

        for MODE in ("full", "lora"):
            _eval_dir_rel = f"../outputs/{ds_code}/eval/{short_name}/stage1_eval/{MODE}_world_model"
            _eval_dir_abs = f"{LF_ROOT}/{_eval_dir_rel[3:]}"
            _adapter_dir  = Path(BASE_DIR) / "outputs" / ds_code / "adapters" / f"{short_name}_stage1_{MODE}"
            _hub_template = f"SaFD-00/{short_name}-{_slug}stage1-{MODE}-world-model-epoch{{epoch}}"

            for epoch in EPOCHS:
                _hub_id = f"SaFD-00/{short_name}-{_slug}stage1-{MODE}-world-model-epoch{epoch}"
                _out = f"{_eval_dir_rel}/epoch-{epoch}"
                print(f"[{model_key}/{_ds_name}][stage1_{MODE}] epoch-{epoch} → {_hub_id}")
                _cmd = (
                    f"mkdir -p {_out} && python scripts/vllm_infer.py "
                    f"--model_name_or_path {_hub_id} "
                    f"--dataset {_cfg['ds_s1_test']} "
                    f"--dataset_dir {LF_ROOT}/data "
                    f"--template {_template} "
                    f"--cutoff_len 8192 "
                    f"--image_max_pixels {_img_px} "
                    f"{_enable_thinking} "
                    f"--vllm_config '{_vllm_cfg}' "
                    f"--save_name {_out}/generated_predictions.jsonl "
                    f"--matrix_save_name {_out}/predict_results.json"
                )
                !{_cmd}
                _score = (
                    f"python {BASE_DIR}/scripts/_hungarian_eval.py score "
                    f"--test {_test_jsonl} "
                    f"--pred {_out}/generated_predictions.jsonl "
                    f"--output {_out}/hungarian_metrics.json"
                )
                !{_score}

            # winner 선정 → adapter 경로의 BEST_CHECKPOINT[.json] (hf_repo_id 필드 포함)
            _select = (
                f"python {BASE_DIR}/scripts/_hungarian_eval.py select "
                f"--eval-dir {_eval_dir_abs} "
                f"--train-dir {_adapter_dir} "
                f"--metric avg_hungarian_f1 "
                f"--hf-repo-template '{_hub_template}'"
            )
            !{_select}


In [ ]:
import json
import re
import math
from collections import Counter
from bs4 import BeautifulSoup
from munkres import Munkres

# ── Hungarian Metric 상수 ─────────────────────────────────────────────────
INTERACTIVE_TAGS = {"button", "input", "a", "select", "textarea"}
CONTENT_TAGS     = {"p", "img", "span"}
CLICKABLE_ATTRS  = {"clickable", "long-clickable"}

W_TAG   = 3.0   # tag 불일치 패널티 (매칭 불가)
W_TEXT  = 1.5   # text 불일치
W_INDEX = 0.2   # DOM index 거리

MATCH_THRESHOLD = 1.5   # 이 이상이면 매칭 거부
INDEX_TAU       = 2     # index_acc: 차이 ≤ τ 이면 위치 정확


# ── 요소 추출 (BeautifulSoup) ─────────────────────────────────────────────

def _collect_texts(el):
    """요소 자신 + 자식 전체에서 텍스트 토큰 수집 (중복 제거, 알파벳순 join)."""
    tokens = set()
    def add(v):
        if v:
            tokens.add(v.strip())
    add(el.get("description"))
    add(el.get("id"))
    for child in el.find_all(True):
        add(child.get("description"))
        add(child.get("id"))
        t = child.get_text(strip=True)
        if t:
            tokens.add(t)
    t = el.get_text(strip=True)
    if t:
        tokens.add(t)
    return " | ".join(sorted(tokens)) if tokens else ""


def _safe_int(v, default=-1):
    try:
        return int(v)
    except (TypeError, ValueError):
        return default


def extract_elements(xml_str):
    """XML/HTML 문자열에서 interactive 요소를 평탄화하여 추출."""
    try:
        soup = BeautifulSoup(xml_str, "xml")
    except Exception:
        soup = BeautifulSoup(xml_str, "html.parser")
    elements = []
    for el in soup.find_all(True):
        tag  = el.name
        idx  = _safe_int(el.get("index", -1))
        text = _collect_texts(el)
        is_interactive = tag in INTERACTIVE_TAGS
        is_content     = (tag in CONTENT_TAGS) and bool(text)
        is_clickable   = any(el.get(a) for a in CLICKABLE_ATTRS)
        if is_interactive or is_content or is_clickable:
            elements.append({"tag": tag, "text": text, "index": idx})
    return elements


# ── 비용 함수 ─────────────────────────────────────────────────────────────

def _text_sim(a, b):
    """Jaccard 유사도 (단어 집합 기준)."""
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    sa = set(a.lower().replace("|", "").split())
    sb = set(b.lower().replace("|", "").split())
    return len(sa & sb) / len(sa | sb)


def _match_cost(e1, e2, max_idx):
    """pred 요소 e1과 gt 요소 e2 사이의 매칭 비용."""
    if e1["tag"] != e2["tag"]:
        return W_TAG
    tc = W_TEXT  * (1.0 - _text_sim(e1["text"], e2["text"]))
    ic = W_INDEX * (abs(e1["index"] - e2["index"]) / max(max_idx, 1))
    return round(tc + ic, 5)


# ── 헝가리안 매칭 ─────────────────────────────────────────────────────────

def _hungarian_match(pred, gt):
    """헝가리안 알고리즘으로 pred-gt 간 최적 1:1 매칭."""
    n, m = len(pred), len(gt)
    if n == 0 or m == 0:
        return [], []
    max_idx = max(
        (e["index"] for e in pred + gt if e["index"] >= 0),
        default=1,
    )
    matrix = [
        [_match_cost(p, g, max_idx) for g in gt]
        for p in pred
    ]
    size   = max(n, m)
    padded = [row + [MATCH_THRESHOLD * 2] * (size - len(row)) for row in matrix]
    while len(padded) < size:
        padded.append([MATCH_THRESHOLD * 2] * size)
    indexes = Munkres().compute(padded)
    pairs = []
    for i, j in indexes:
        if i < n and j < m and matrix[i][j] < MATCH_THRESHOLD:
            pairs.append((i, j, matrix[i][j]))
    return pairs, matrix


def compute_hungarian_acc(pred_str, gt_str):
    """모델 생성 XML과 정답 XML을 비교해 hungarian 기반 평가 메트릭을 반환."""
    _zero = {
        "hungarian_ea": 0.0, "hungarian_f1": 0.0,
        "hungarian_prec": 0.0, "hungarian_rec": 0.0,
        "hungarian_text": 0.0, "hungarian_idx": 0.0,
    }
    try:
        pred_els = extract_elements(pred_str)
        gt_els   = extract_elements(gt_str)
    except Exception:
        return _zero
    if not gt_els:
        return _zero

    pairs, _ = _hungarian_match(pred_els, gt_els)
    n_pred, n_gt, n_matched = len(pred_els), len(gt_els), len(pairs)

    ea   = n_matched / max(n_pred, n_gt) if max(n_pred, n_gt) > 0 else 0.0
    prec = n_matched / n_pred             if n_pred  > 0           else 0.0
    rec  = n_matched / n_gt               if n_gt    > 0           else 0.0
    f1   = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0    else 0.0

    if pairs:
        text_sims = [_text_sim(pred_els[i]["text"], gt_els[j]["text"]) for i, j, _ in pairs]
        idx_diffs = [abs(pred_els[i]["index"] - gt_els[j]["index"]) for i, j, _ in pairs]
        text_avg  = sum(text_sims) / len(text_sims)
        idx_acc   = sum(1 for d in idx_diffs if d <= INDEX_TAU) / len(idx_diffs)
    else:
        text_avg = 0.0
        idx_acc  = 0.0

    return {
        "hungarian_ea":   round(ea, 4),
        "hungarian_f1":   round(f1, 4),
        "hungarian_prec": round(prec, 4),
        "hungarian_rec":  round(rec, 4),
        "hungarian_text": round(text_avg, 4),
        "hungarian_idx":  round(idx_acc, 4),
    }


# ── 텍스트 생성 품질 메트릭 ───────────────────────────────────────────────

def calc_bleu(reference, hypothesis, max_n=4):
    """BLEU-4 score를 계산합니다."""
    ref_tokens = reference.split()
    hyp_tokens = hypothesis.split()

    if not hyp_tokens or not ref_tokens:
        return 0.0

    bp = min(1.0, math.exp(1 - len(ref_tokens) / len(hyp_tokens)))

    precisions = []
    for n in range(1, max_n + 1):
        ref_ngrams = Counter(tuple(ref_tokens[i:i+n]) for i in range(len(ref_tokens) - n + 1))
        hyp_ngrams = Counter(tuple(hyp_tokens[i:i+n]) for i in range(len(hyp_tokens) - n + 1))

        clipped = sum(min(count, ref_ngrams.get(ng, 0)) for ng, count in hyp_ngrams.items())
        total = sum(hyp_ngrams.values())

        if total == 0:
            precisions.append(0)
        else:
            precisions.append(clipped / total)

    if any(p == 0 for p in precisions):
        return 0.0

    log_avg = sum(math.log(p) for p in precisions) / max_n
    return bp * math.exp(log_avg)


def calc_rouge_l(reference, hypothesis):
    """ROUGE-L (F1) score를 계산합니다."""
    ref_tokens = reference.split()
    hyp_tokens = hypothesis.split()

    if not ref_tokens or not hyp_tokens:
        return 0.0

    m, n = len(ref_tokens), len(hyp_tokens)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if ref_tokens[i-1] == hyp_tokens[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])

    lcs_len = dp[m][n]
    precision = lcs_len / n
    recall = lcs_len / m

    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


# ── Stage 1 전체 평가 함수 ────────────────────────────────────────────────

def evaluate_stage1_predictions(test_path, pred_path):
    """Stage 1 prediction 전체 평가를 수행합니다."""
    with open(test_path, 'r') as f:
        gt_entries = [json.loads(line) for line in f if line.strip()]
    with open(pred_path, 'r') as f:
        pred_entries = [json.loads(line) for line in f if line.strip()]

    results = []
    for gt_entry, pred_entry in zip(gt_entries, pred_entries):
        gt_text = gt_entry['messages'][-1]['value']
        pred_text = pred_entry.get('predict', pred_entry.get('output', ''))

        bleu = calc_bleu(gt_text, pred_text)
        rouge_l = calc_rouge_l(gt_text, pred_text)
        exact_match = 1.0 if gt_text.strip() == pred_text.strip() else 0.0
        hungarian = compute_hungarian_acc(pred_text, gt_text)

        results.append({
            'bleu': bleu, 'rouge_l': rouge_l,
            'exact_match': exact_match, 'hungarian': hungarian,
        })

    total = len(results)
    metrics = {
        'total': total,
        'avg_bleu': sum(r['bleu'] for r in results) / total if total else 0,
        'avg_rouge_l': sum(r['rouge_l'] for r in results) / total if total else 0,
        'exact_match_rate': sum(r['exact_match'] for r in results) / total if total else 0,
        'avg_hungarian_ea':   sum(r['hungarian']['hungarian_ea'] for r in results) / total if total else 0,
        'avg_hungarian_f1':   sum(r['hungarian']['hungarian_f1'] for r in results) / total if total else 0,
        'avg_hungarian_prec': sum(r['hungarian']['hungarian_prec'] for r in results) / total if total else 0,
        'avg_hungarian_rec':  sum(r['hungarian']['hungarian_rec'] for r in results) / total if total else 0,
        'avg_hungarian_text': sum(r['hungarian']['hungarian_text'] for r in results) / total if total else 0,
        'avg_hungarian_idx':  sum(r['hungarian']['hungarian_idx'] for r in results) / total if total else 0,
    }
    return metrics, results

print("[Loaded] Stage 1 evaluation functions: calc_bleu, calc_rouge_l, compute_hungarian_acc, evaluate_stage1_predictions")

In [ ]:
import json
import math
import pandas as pd
from pathlib import Path

all_stage1_metrics = {}

for model_key, ds_configs in CONFIGS.items():
    for ds_name, cfg in ds_configs.items():
        short_name = cfg["short_name"]
        op = cfg["output_prefix"]
        s1_eval_dir = Path(BASE_DIR) / "outputs" / op.rstrip("/") / "eval" / short_name / "stage1_eval"
        hung_dir = s1_eval_dir / "hungarian_matching"
        test_path = Path(cfg["data_dir"]) / "gui-model_stage1_test.jsonl"
        train_dir = Path(BASE_DIR) / cfg["save_s1"].lstrip("../")

        MODEL_PREDS = {}
        base_pred = hung_dir / "base" / "generated_predictions.jsonl"
        if base_pred.exists():
            MODEL_PREDS["Baseline (Zero-shot)"] = base_pred
        for p in sorted(hung_dir.glob("epoch-*/generated_predictions.jsonl"),
                        key=lambda x: int(x.parent.name.split("-")[1])):
            MODEL_PREDS[p.parent.name] = p

        combo_key = f"{model_key}/{ds_name}"
        print(f"\n{'='*70}")
        print(f"=== {combo_key} Stage 1 Evaluation ({len(MODEL_PREDS)} model(s)) ===")

        base_loss_path = s1_eval_dir / "eval_loss" / "base" / "eval_results.json"
        fwm_loss_path  = s1_eval_dir / "eval_loss" / "full_world_model" / "eval_results.json"
        if base_loss_path.exists() and fwm_loss_path.exists():
            with open(base_loss_path, 'r') as f: base_loss_metrics = json.load(f)
            with open(fwm_loss_path,  'r') as f: fwm_loss_metrics  = json.load(f)
            print(f"\n  Loss-based Evaluation:")
            print(f"    Base     eval_loss: {base_loss_metrics['eval_loss']:.4f}  perplexity: {math.exp(base_loss_metrics['eval_loss']):.4f}")
            print(f"    FWM      eval_loss: {fwm_loss_metrics['eval_loss']:.4f}  perplexity: {math.exp(fwm_loss_metrics['eval_loss']):.4f}")

        ds_metrics = {}
        for model_name, pred_path in MODEL_PREDS.items():
            if not pred_path.exists():
                print(f"  [SKIP] {model_name}: {pred_path} missing")
                continue
            metrics, _ = evaluate_stage1_predictions(str(test_path), str(pred_path))
            ds_metrics[model_name] = metrics
            out_metric_path = pred_path.parent / "hungarian_metrics.json"
            with open(out_metric_path, 'w', encoding='utf-8') as f:
                json.dump(metrics, f, ensure_ascii=False, indent=2)

        all_stage1_metrics[combo_key] = ds_metrics

        if ds_metrics:
            print(f"\n  === {combo_key} Stage 1 Comparison ===")
            comparison = pd.DataFrame({
                name: {
                    'BLEU-4': f"{m['avg_bleu']:.4f}", 'ROUGE-L': f"{m['avg_rouge_l']:.4f}",
                    'Exact Match': f"{m['exact_match_rate']:.1%}", 'Hungarian EA': f"{m['avg_hungarian_ea']:.4f}",
                    'Hungarian F1': f"{m['avg_hungarian_f1']:.4f}", 'Hungarian Prec': f"{m['avg_hungarian_prec']:.4f}",
                    'Hungarian Rec': f"{m['avg_hungarian_rec']:.4f}", 'Hungarian Text': f"{m['avg_hungarian_text']:.4f}",
                    'Hungarian Idx': f"{m['avg_hungarian_idx']:.4f}",
                } for name, m in ds_metrics.items()
            })
            display(comparison)

            ckpt_metrics = {k: v for k, v in ds_metrics.items() if k.startswith("checkpoint-")}
            if ckpt_metrics:
                best_name, best_m = max(ckpt_metrics.items(),
                                        key=lambda kv: (kv[1]['avg_hungarian_f1'],
                                                        int(kv[0].split('-')[1])))
                train_dir.mkdir(parents=True, exist_ok=True)
                (train_dir / "BEST_CHECKPOINT").write_text(best_name + "\n", encoding='utf-8')
                summary = {
                    "selected": best_name,
                    "metric": "avg_hungarian_f1",
                    "metric_value": best_m['avg_hungarian_f1'],
                    "candidates": [
                        {"checkpoint": k, "avg_hungarian_f1": v['avg_hungarian_f1'],
                         "avg_bleu": v['avg_bleu'], "avg_rouge_l": v['avg_rouge_l'],
                         "exact_match_rate": v['exact_match_rate']}
                        for k, v in ckpt_metrics.items()
                    ],
                }
                (train_dir / "BEST_CHECKPOINT.json").write_text(
                    json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding='utf-8')
                print(f"\n  [{combo_key}] Hungarian F1 winner: {best_name} (F1={best_m['avg_hungarian_f1']:.4f})")
                print(f"     -> {train_dir / 'BEST_CHECKPOINT'}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

for combo_key, ds_metrics in all_stage1_metrics.items():
    if not ds_metrics:
        continue
    model_key, ds_name = combo_key.split("/")
    cfg = CONFIGS[model_key][ds_name]
    short_name = cfg["short_name"]
    model_names = list(ds_metrics.keys())
    n_models = len(model_names)
    colors = ['#9E9E9E', '#FF5722', '#2196F3', '#4CAF50'][:n_models]

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))

    text_metrics = ['avg_bleu', 'avg_rouge_l', 'exact_match_rate']
    text_labels = ['BLEU-4', 'ROUGE-L', 'Exact Match']
    x1 = np.arange(len(text_metrics))
    width = 0.8 / n_models

    for i, name in enumerate(model_names):
        values = [ds_metrics[name][m] for m in text_metrics]
        offset = (i - n_models / 2 + 0.5) * width
        axes[0].bar(x1 + offset, values, width, label=name, color=colors[i])

    axes[0].set_xlabel('Metric'); axes[0].set_ylabel('Score')
    axes[0].set_title('Text Generation Metrics')
    axes[0].set_xticks(x1); axes[0].set_xticklabels(text_labels, rotation=15)
    axes[0].legend(fontsize=8); axes[0].set_ylim(0, 1.0); axes[0].grid(axis='y', alpha=0.3)

    hung_metrics = ['avg_hungarian_ea', 'avg_hungarian_f1', 'avg_hungarian_prec',
                    'avg_hungarian_rec', 'avg_hungarian_text', 'avg_hungarian_idx']
    hung_labels = ['EA', 'F1', 'Precision', 'Recall', 'Text Sim', 'Index Acc']
    x2 = np.arange(len(hung_metrics))

    for i, name in enumerate(model_names):
        values = [ds_metrics[name][m] for m in hung_metrics]
        offset = (i - n_models / 2 + 0.5) * width
        axes[1].bar(x2 + offset, values, width, label=name, color=colors[i])

    axes[1].set_xlabel('Metric'); axes[1].set_ylabel('Score')
    axes[1].set_title('Hungarian Element Matching')
    axes[1].set_xticks(x2); axes[1].set_xticklabels(hung_labels, rotation=15)
    axes[1].legend(fontsize=8); axes[1].set_ylim(0, 1.0); axes[1].grid(axis='y', alpha=0.3)

    plt.suptitle(f'Stage 1 Evaluation ({combo_key})', fontsize=14, fontweight='bold')
    plt.tight_layout()

    op = cfg["output_prefix"]
    s1_eval_dir = Path(BASE_DIR) / "outputs" / op.rstrip("/") / "eval" / short_name / "stage1_eval"
    save_path = str(s1_eval_dir / "hungarian_matching" / "stage1_hungarian_matching_evaluation.png")
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"[Saved] {save_path}")

In [ ]:
import json, math
from pathlib import Path
from datetime import datetime

# Stage 1 Hungarian Matching report — baseline + per-checkpoint sweep.
# 사전 준비: all_stage1_metrics[f"{model_key}/{ds_name}"] 은 다음 구조여야 한다.
#   {
#     "base":                 <metrics dict>,
#     "full_world_model":     {"<ckpt>": <metrics dict>, ...},   # optional
#     "lora_world_model":     {"<ckpt>": <metrics dict>, ...},   # optional
#   }
# (Cell 57 이 per-checkpoint hungarian_metrics.json 을 이 구조로 적재하도록 갱신됨.)

def _winner_ckpt(adapter_dir: Path, fallback_key: str):
    bc = adapter_dir / "BEST_CHECKPOINT"
    if bc.exists():
        return bc.read_text().strip()
    return fallback_key

def _metric_row(label: str, m: dict, keys):
    cells = [f"{m[key]:{fmt}}" if key in m else "-" for key, _lbl, fmt in keys]
    return f"| {label} | " + " | ".join(cells) + " |"

METRIC_KEYS = [
    ("avg_bleu",          "BLEU-4",         ".4f"),
    ("avg_rouge_l",       "ROUGE-L",        ".4f"),
    ("exact_match_rate",  "Exact Match",    ".1%"),
    ("avg_hungarian_ea",  "Hungarian EA",   ".4f"),
    ("avg_hungarian_f1",  "Hungarian F1",   ".4f"),
    ("avg_hungarian_prec","Hungarian Prec", ".4f"),
    ("avg_hungarian_rec", "Hungarian Rec",  ".4f"),
    ("avg_hungarian_text","Hungarian Text", ".4f"),
    ("avg_hungarian_idx", "Hungarian Idx",  ".4f"),
]

for combo_key, entries in all_stage1_metrics.items():
    model_key, ds_name = combo_key.split("/")
    cfg = CONFIGS[model_key][ds_name]
    short_name = cfg["short_name"]
    ds_code    = cfg["output_prefix"].rstrip("/")
    s1_eval_dir = Path(BASE_DIR) / "outputs" / ds_code / "eval" / short_name / "stage1_eval"
    base_metrics = entries.get("base")

    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    rlines = [f"# Stage 1 Hungarian Matching Report ({model_key} / {ds_name})",
              f"\n> Generated: {now}\n",
              "## Experiment Setup\n",
              "| Item | Value |",
              "|------|-------|",
              f"| Model | {cfg['model_id']} |",
              f"| Dataset | {ds_name} |",
              f"| Training | {cfg['stage1']['epochs']} epochs, LR={cfg['stage1']['lr']} (cosine) |",
              ""]
    if base_metrics:
        rlines.append("## Baseline (Zero-shot)\n")
        rlines += ["| Metric | Score |", "|--------|-------|"]
        for key, lbl, fmt in METRIC_KEYS:
            if key in base_metrics:
                rlines.append(f"| {lbl} | {base_metrics[key]:{fmt}} |")
        rlines.append("")

    for MODE in ("full", "lora"):
        ckpt_map = entries.get(f"{MODE}_world_model")
        if not ckpt_map:
            continue
        adapter_dir = Path(BASE_DIR) / "outputs" / ds_code / "adapters" / f"{short_name}_stage1_{MODE}"
        winner = _winner_ckpt(adapter_dir, max(ckpt_map.keys(),
                              key=lambda c: int(c.split("-")[-1]) if c.split("-")[-1].isdigit() else -1))
        rlines.append(f"## World-Model (stage1_{MODE}) — per-checkpoint sweep\n")
        header = ["Checkpoint"] + [lbl for _, lbl, _ in METRIC_KEYS]
        rlines.append("| " + " | ".join(header) + " |")
        rlines.append("|" + "|".join(["--------"] * len(header)) + "|")
        for name, m in sorted(ckpt_map.items(),
                              key=lambda kv: int(kv[0].split("-")[-1]) if kv[0].split("-")[-1].isdigit() else -1):
            mark = "  **(winner)**" if name == winner else ""
            row = [name + mark] + [f"{m[key]:{fmt}}" if key in m else "-" for key, _, fmt in METRIC_KEYS]
            rlines.append("| " + " | ".join(row) + " |")
        rlines.append("")

    report = "\n".join(rlines)
    report_path = s1_eval_dir / "hungarian_matching" / "evaluation_report.md"
    report_path.parent.mkdir(parents=True, exist_ok=True)
    report_path.write_text(report, encoding="utf-8")
    print(f"[Saved] {report_path}")

print("\nDone.")


## 6. Stage 2 SFT Training

**핵심 가설**: GUI World Modeling으로 사전학습된 모델은 동일 베이스 대비 Action Prediction SFT에서 더 높은 성능을 보인다.

| ID | 역할 | 모델 | MobiBench HF ID 패턴 | AndroidControl HF ID 패턴 |
|----|------|------|---------------------|---------------------------|
| [stage2] | Base | 각 모델의 원본 | `{model_id}` (예: `Qwen/Qwen3-VL-8B-Instruct`) | 동일 |
| [stage1+stage2] | World Model | Stage 1 Merged | `SaFD-00/{model}-mb-stage1-world-model` | `SaFD-00/{model}-ac-stage1-world-model` |

> `{model}` = 모델 short_name, `{model_id}` = Cell 3 `_MODEL_CONFIG` 의 원본 HuggingFace ID

**Base 비교:** 각 모델의 원본 베이스 (Zero-shot, 학습 없음)도 함께 평가

**핵심 비교:**
- stage2 vs stage1+stage2 → World Modeling 사전학습이 Action Prediction에 미치는 효과

**공통 학습 설정:**

| 항목 | MobiBench | AndroidControl |
|------|-----------|----------------|
| Method | LoRA (r=16, α=32, dropout=0.1) | LoRA (r=32, α=64, dropout=0.1) |
| Dataset | GUI-Model-MB_stage2_train (~2,987건) | GUI-Model-AC_stage2_train (~87,090건) |
| Effective Batch | 32 (2 × 4 × 4 GPU) | 64 (2 × 8 × N_GPU) |
| Eval Batch (per-device) | 1 | 4 |
| Learning Rate | 3e-5 (cosine, warmup=0.05) | 5e-5 (cosine, warmup=0.03) |
| Epochs | 5 | 3 |
| Save/Eval Strategy | epoch | epoch |
| Weight Decay | 0.01 | 동일 |
| max_grad_norm | 1.0 | 동일 |
| Precision | bf16 | 동일 |
| 학습 중 eval | 비활성화 (post-training checkpoint sweep 으로 Overall Score winner 선택) | 동일 |

> AC LoRA rank/alpha 상향(r=16→32, α=32→64)은 50K-100K 샘플 규모에 맞춘 capacity 확장입니다 (vlm-gui-finetuning-research.md 권장 64-128의 보수 선택).

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2-VL-2B] [MobiBench] [stage2] Stage 2 - Base
!DISABLE_VERSION_CHECK=1 llamafactory-cli train custom/GUI-Model-MB/stage2_lora/qwen2-vl-2b_base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2-VL-2B] [AndroidControl] [stage2] Stage 2 - Base
!DISABLE_VERSION_CHECK=1 llamafactory-cli train custom/GUI-Model-AC/stage2_lora/qwen2-vl-2b_base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2-VL-7B] [MobiBench] [stage2] Stage 2 - Base
!DISABLE_VERSION_CHECK=1 llamafactory-cli train custom/GUI-Model-MB/stage2_lora/qwen2-vl-7b_base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2-VL-7B] [AndroidControl] [stage2] Stage 2 - Base
!DISABLE_VERSION_CHECK=1 llamafactory-cli train custom/GUI-Model-AC/stage2_lora/qwen2-vl-7b_base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2.5-VL-3B] [MobiBench] [stage2] Stage 2 - Base
!DISABLE_VERSION_CHECK=1 llamafactory-cli train custom/GUI-Model-MB/stage2_lora/qwen2.5-vl-3b_base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2.5-VL-3B] [AndroidControl] [stage2] Stage 2 - Base
!DISABLE_VERSION_CHECK=1 llamafactory-cli train custom/GUI-Model-AC/stage2_lora/qwen2.5-vl-3b_base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2.5-VL-7B] [MobiBench] [stage2] Stage 2 - Base
!DISABLE_VERSION_CHECK=1 llamafactory-cli train custom/GUI-Model-MB/stage2_lora/qwen2.5-vl-7b_base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2.5-VL-7B] [AndroidControl] [stage2] Stage 2 - Base
!DISABLE_VERSION_CHECK=1 llamafactory-cli train custom/GUI-Model-AC/stage2_lora/qwen2.5-vl-7b_base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen3-VL-2B] [MobiBench] [stage2] Stage 2 - Base
!DISABLE_VERSION_CHECK=1 llamafactory-cli train custom/GUI-Model-MB/stage2_lora/qwen3-vl-2b_base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen3-VL-2B] [AndroidControl] [stage2] Stage 2 - Base
!DISABLE_VERSION_CHECK=1 llamafactory-cli train custom/GUI-Model-AC/stage2_lora/qwen3-vl-2b_base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen3-VL-4B] [MobiBench] [stage2] Stage 2 - Base
!DISABLE_VERSION_CHECK=1 llamafactory-cli train custom/GUI-Model-MB/stage2_lora/qwen3-vl-4b_base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen3-VL-4B] [AndroidControl] [stage2] Stage 2 - Base
!DISABLE_VERSION_CHECK=1 llamafactory-cli train custom/GUI-Model-AC/stage2_lora/qwen3-vl-4b_base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen3-VL-8B] [MobiBench] [stage2] Stage 2 - Base
!DISABLE_VERSION_CHECK=1 llamafactory-cli train custom/GUI-Model-MB/stage2_lora/qwen3-vl-8b_base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen3-VL-8B] [AndroidControl] [stage2] Stage 2 - Base
!DISABLE_VERSION_CHECK=1 llamafactory-cli train custom/GUI-Model-AC/stage2_lora/qwen3-vl-8b_base.yaml

In [ ]:
os.chdir(BASE_DIR)

## [Gemma-4-E2B] [MobiBench] [stage2] Stage 2 - Base | Unsloth
!cd "$BASE_DIR" && accelerate launch --multi_gpu --num_processes ${NPROC_PER_NODE:-2} \
    scripts/_unsloth_train.py \
    --config unsloth/configs/GUI-Model-MB/stage2_lora/gemma-4-e2b_base.yaml \
    --base-dir "$BASE_DIR"


In [ ]:
os.chdir(BASE_DIR)

## [Gemma-4-E2B] [AndroidControl] [stage2] Stage 2 - Base | Unsloth
!cd "$BASE_DIR" && accelerate launch --multi_gpu --num_processes ${NPROC_PER_NODE:-2} \
    scripts/_unsloth_train.py \
    --config unsloth/configs/GUI-Model-AC/stage2_lora/gemma-4-e2b_base.yaml \
    --base-dir "$BASE_DIR"


In [ ]:
os.chdir(BASE_DIR)

## [Gemma-4-E4B] [MobiBench] [stage2] Stage 2 - Base | Unsloth
!cd "$BASE_DIR" && accelerate launch --multi_gpu --num_processes ${NPROC_PER_NODE:-2} \
    scripts/_unsloth_train.py \
    --config unsloth/configs/GUI-Model-MB/stage2_lora/gemma-4-e4b_base.yaml \
    --base-dir "$BASE_DIR"


In [ ]:
os.chdir(BASE_DIR)

## [Gemma-4-E4B] [AndroidControl] [stage2] Stage 2 - Base | Unsloth
!cd "$BASE_DIR" && accelerate launch --multi_gpu --num_processes ${NPROC_PER_NODE:-2} \
    scripts/_unsloth_train.py \
    --config unsloth/configs/GUI-Model-AC/stage2_lora/gemma-4-e4b_base.yaml \
    --base-dir "$BASE_DIR"


In [ ]:
os.chdir(LF_ROOT)

## [LLaVA-v1.6-Mistral-7B] [MobiBench] [stage2] Stage 2 - Base
!DISABLE_VERSION_CHECK=1 llamafactory-cli train custom/GUI-Model-MB/stage2_lora/llava-v1.6-mistral-7b_base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [LLaVA-v1.6-Mistral-7B] [AndroidControl] [stage2] Stage 2 - Base
!DISABLE_VERSION_CHECK=1 llamafactory-cli train custom/GUI-Model-AC/stage2_lora/llava-v1.6-mistral-7b_base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [LLaVA-v1.6-Vicuna-7B] [MobiBench] [stage2] Stage 2 - Base
!DISABLE_VERSION_CHECK=1 llamafactory-cli train custom/GUI-Model-MB/stage2_lora/llava-v1.6-vicuna-7b_base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [LLaVA-v1.6-Vicuna-7B] [AndroidControl] [stage2] Stage 2 - Base
!DISABLE_VERSION_CHECK=1 llamafactory-cli train custom/GUI-Model-AC/stage2_lora/llava-v1.6-vicuna-7b_base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [LLaMA3-LLaVA-Next-8B] [MobiBench] [stage2] Stage 2 - Base
!DISABLE_VERSION_CHECK=1 llamafactory-cli train custom/GUI-Model-MB/stage2_lora/llama3-llava-next-8b_base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [LLaMA3-LLaVA-Next-8B] [AndroidControl] [stage2] Stage 2 - Base
!DISABLE_VERSION_CHECK=1 llamafactory-cli train custom/GUI-Model-AC/stage2_lora/llama3-llava-next-8b_base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2-VL-2B] [MobiBench] [stage1+stage2] Stage 2 - World Model
!DISABLE_VERSION_CHECK=1 llamafactory-cli train custom/GUI-Model-MB/stage2_lora/qwen2-vl-2b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2-VL-2B] [AndroidControl] [stage1+stage2] Stage 2 - World Model
!DISABLE_VERSION_CHECK=1 llamafactory-cli train custom/GUI-Model-AC/stage2_lora/qwen2-vl-2b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2-VL-7B] [MobiBench] [stage1+stage2] Stage 2 - World Model
!DISABLE_VERSION_CHECK=1 llamafactory-cli train custom/GUI-Model-MB/stage2_lora/qwen2-vl-7b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2-VL-7B] [AndroidControl] [stage1+stage2] Stage 2 - World Model
!DISABLE_VERSION_CHECK=1 llamafactory-cli train custom/GUI-Model-AC/stage2_lora/qwen2-vl-7b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2.5-VL-3B] [MobiBench] [stage1+stage2] Stage 2 - World Model
!DISABLE_VERSION_CHECK=1 llamafactory-cli train custom/GUI-Model-MB/stage2_lora/qwen2.5-vl-3b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2.5-VL-3B] [AndroidControl] [stage1+stage2] Stage 2 - World Model
!DISABLE_VERSION_CHECK=1 llamafactory-cli train custom/GUI-Model-AC/stage2_lora/qwen2.5-vl-3b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2.5-VL-7B] [MobiBench] [stage1+stage2] Stage 2 - World Model
!DISABLE_VERSION_CHECK=1 llamafactory-cli train custom/GUI-Model-MB/stage2_lora/qwen2.5-vl-7b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2.5-VL-7B] [AndroidControl] [stage1+stage2] Stage 2 - World Model
!DISABLE_VERSION_CHECK=1 llamafactory-cli train custom/GUI-Model-AC/stage2_lora/qwen2.5-vl-7b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen3-VL-2B] [MobiBench] [stage1+stage2] Stage 2 - World Model
!DISABLE_VERSION_CHECK=1 llamafactory-cli train custom/GUI-Model-MB/stage2_lora/qwen3-vl-2b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen3-VL-2B] [AndroidControl] [stage1+stage2] Stage 2 - World Model
!DISABLE_VERSION_CHECK=1 llamafactory-cli train custom/GUI-Model-AC/stage2_lora/qwen3-vl-2b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen3-VL-4B] [MobiBench] [stage1+stage2] Stage 2 - World Model
!DISABLE_VERSION_CHECK=1 llamafactory-cli train custom/GUI-Model-MB/stage2_lora/qwen3-vl-4b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen3-VL-4B] [AndroidControl] [stage1+stage2] Stage 2 - World Model
!DISABLE_VERSION_CHECK=1 llamafactory-cli train custom/GUI-Model-AC/stage2_lora/qwen3-vl-4b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen3-VL-8B] [MobiBench] [stage1+stage2] Stage 2 - World Model
!DISABLE_VERSION_CHECK=1 llamafactory-cli train custom/GUI-Model-MB/stage2_lora/qwen3-vl-8b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen3-VL-8B] [AndroidControl] [stage1+stage2] Stage 2 - World Model
!DISABLE_VERSION_CHECK=1 llamafactory-cli train custom/GUI-Model-AC/stage2_lora/qwen3-vl-8b_world-model.yaml

In [ ]:
os.chdir(BASE_DIR)

## [Gemma-4-E2B] [MobiBench] [stage1+stage2] Stage 2 - World Model | Unsloth
!cd "$BASE_DIR" && accelerate launch --multi_gpu --num_processes ${NPROC_PER_NODE:-2} \
    scripts/_unsloth_train.py \
    --config unsloth/configs/GUI-Model-MB/stage2_lora/gemma-4-e2b_world-model.yaml \
    --base-dir "$BASE_DIR"


In [ ]:
os.chdir(BASE_DIR)

## [Gemma-4-E2B] [AndroidControl] [stage1+stage2] Stage 2 - World Model | Unsloth
!cd "$BASE_DIR" && accelerate launch --multi_gpu --num_processes ${NPROC_PER_NODE:-2} \
    scripts/_unsloth_train.py \
    --config unsloth/configs/GUI-Model-AC/stage2_lora/gemma-4-e2b_world-model.yaml \
    --base-dir "$BASE_DIR"


In [ ]:
os.chdir(BASE_DIR)

## [Gemma-4-E4B] [MobiBench] [stage1+stage2] Stage 2 - World Model | Unsloth
!cd "$BASE_DIR" && accelerate launch --multi_gpu --num_processes ${NPROC_PER_NODE:-2} \
    scripts/_unsloth_train.py \
    --config unsloth/configs/GUI-Model-MB/stage2_lora/gemma-4-e4b_world-model.yaml \
    --base-dir "$BASE_DIR"


In [ ]:
os.chdir(BASE_DIR)

## [Gemma-4-E4B] [AndroidControl] [stage1+stage2] Stage 2 - World Model | Unsloth
!cd "$BASE_DIR" && accelerate launch --multi_gpu --num_processes ${NPROC_PER_NODE:-2} \
    scripts/_unsloth_train.py \
    --config unsloth/configs/GUI-Model-AC/stage2_lora/gemma-4-e4b_world-model.yaml \
    --base-dir "$BASE_DIR"


In [ ]:
os.chdir(LF_ROOT)

## [LLaVA-v1.6-Mistral-7B] [MobiBench] [stage1+stage2] Stage 2 - World Model
!DISABLE_VERSION_CHECK=1 llamafactory-cli train custom/GUI-Model-MB/stage2_lora/llava-v1.6-mistral-7b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [LLaVA-v1.6-Mistral-7B] [AndroidControl] [stage1+stage2] Stage 2 - World Model
!DISABLE_VERSION_CHECK=1 llamafactory-cli train custom/GUI-Model-AC/stage2_lora/llava-v1.6-mistral-7b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [LLaVA-v1.6-Vicuna-7B] [MobiBench] [stage1+stage2] Stage 2 - World Model
!DISABLE_VERSION_CHECK=1 llamafactory-cli train custom/GUI-Model-MB/stage2_lora/llava-v1.6-vicuna-7b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [LLaVA-v1.6-Vicuna-7B] [AndroidControl] [stage1+stage2] Stage 2 - World Model
!DISABLE_VERSION_CHECK=1 llamafactory-cli train custom/GUI-Model-AC/stage2_lora/llava-v1.6-vicuna-7b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [LLaMA3-LLaVA-Next-8B] [MobiBench] [stage1+stage2] Stage 2 - World Model
!DISABLE_VERSION_CHECK=1 llamafactory-cli train custom/GUI-Model-MB/stage2_lora/llama3-llava-next-8b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [LLaMA3-LLaVA-Next-8B] [AndroidControl] [stage1+stage2] Stage 2 - World Model
!DISABLE_VERSION_CHECK=1 llamafactory-cli train custom/GUI-Model-AC/stage2_lora/llama3-llava-next-8b_world-model.yaml

## 7. Stage 2 Merge & Upload to HuggingFace (전 epoch push)

> **실행 전 필수**: Section 7 의 Stage 2 평가를 완료한 뒤 아래 **Stage 2 Merge YAML** 셀을 재실행하여 각 variant 의 `BEST_CHECKPOINT` 가 반영된 Merge YAML 로 덮어쓰세요. 그 후 아래 export cell 두 개를 실행합니다. Merge 결과물은 `outputs/{MODEL}/{DS}/stage2_lora_{base,world_model}/` 에 저장됩니다.


Stage 2 LoRA 어댑터를 베이스 모델에 병합(merge)하고 HuggingFace Hub에 업로드합니다. **어댑터는 Stage 2 평가 단계에서 선택된 winner 체크포인트** 를 사용합니다 (`saves/{MODEL}/{DS}/stage2_lora/{lora_base,lora_world_model}/BEST_CHECKPOINT` 파일 기준). World Model variant 는 HF Hub 가 아닌 **로컬 경로** (`outputs/{MODEL}/{DS}/stage1_merged`)를 사용해 네트워크 의존을 제거합니다.

**선택 방식:**
1. Stage 2 학습 결과 `saves/{MODEL}/{DS}/stage2_lora/{variant}/checkpoint-*/` 다수 생성
2. Stage 2 평가 파이프라인이 각 체크포인트별 Action 메트릭 계산 → `BEST_CHECKPOINT` 파일 기록
3. 아래 **Stage 2 Merge YAML** 셀이 그 파일을 읽어 merge YAML 의 `adapter_name_or_path` 를 winner 로 설정
4. `BEST_CHECKPOINT` 없는 모델/데이터셋 조합은 `[SKIP]` 경고와 함께 건너뜀 — stage2 평가 완료 후 Merge YAML 셀 재실행

**Merge 대상 (MobiBench, 모델별):**

| ID | Base Model | Adapter | Export (HF Hub) |
|----|------------|---------|------------------|
| [stage2]        | `{model_id}` (원본)                    | `saves/{MODEL}/MB/stage2_lora/lora_base/<winner>` | `SaFD-00/{model}-mb-stage2-base` |
| [stage1+stage2] | `outputs/{MODEL}/MB/stage1_merged` (로컬) | `saves/{MODEL}/MB/stage2_lora/lora_world_model/<winner>` | `SaFD-00/{model}-mb-stage2-world-model` |

**Merge 대상 (AndroidControl, 모델별):**

| ID | Base Model | Adapter | Export (HF Hub) |
|----|------------|---------|------------------|
| [stage2]        | `{model_id}` (원본)                    | `saves/{MODEL}/AC/stage2_lora/lora_base/<winner>` | `SaFD-00/{model}-ac-stage2-base` |
| [stage1+stage2] | `outputs/{MODEL}/AC/stage1_merged` (로컬) | `saves/{MODEL}/AC/stage2_lora/lora_world_model/<winner>` | `SaFD-00/{model}-ac-stage2-world-model` |

> `{model}` = 모델 short_name, `{model_id}` = Cell 3 `_MODEL_CONFIG` 의 원본 HuggingFace ID (예: `Qwen/Qwen3-VL-8B-Instruct`)

> **주의:** Merge 시 quantized 모델이나 quantization_bit를 사용하지 마세요. 아래 **Stage 2 Merge YAML** 셀은 **Stage 2 평가(winner 선택)** 이후에 재실행해야 올바른 YAML 이 생성됩니다.

> **흐름**: `stage2_train.sh` → `stage2_merge.sh` (variant × 전 epoch 를 각각 merge + HF push) → `stage2_eval.sh` (HF pull sweep). Stage 2 world-model variant 의 S1 base 는 `adapters/.../BEST_CHECKPOINT.json.epoch` 에서 winner 를 읽어 `merged/.../epoch-{Ewin}/` 를 로컬 로드.


#### [LlamaFactory] Stage 2 Merge YAML

`LlamaFactory/examples/custom/GUI-Model-{MB,AC}/stage2_merge/{MODEL}_{base,world-model}.yaml` 생성. `outputs/{DS}/adapters/{MODEL}_stage2_lora_{base,world_model}/BEST_CHECKPOINT` 파일을 읽어 Overall Score winner 어댑터를 `adapter_name_or_path` 에 주입. 두 variant 중 하나라도 파일이 없으면 해당 조합을 `[SKIP]` 경고 후 건너뛰며, 평가 완료 후 재실행하면 건너뛴 항목도 생성. Merge 결과물은 `outputs/{DS}/merged/{MODEL}_stage2_lora_{base,world_model}/` 에 저장. `backend == "llamafactory"` 모델만 대상 (qwen / llava 계열).


In [ ]:
from pathlib import Path

_created, _skipped = 0, 0

for model_key, ds_configs in CONFIGS.items():
    for ds_name, cfg in ds_configs.items():
        if cfg["model_config"].get("backend") == "unsloth":
            continue
        lf_sub = cfg["lf_subfolder"]
        yaml_dir = Path(LF_ROOT) / "examples" / "custom" / lf_sub / "stage2_merge"
        yaml_dir.mkdir(parents=True, exist_ok=True)

        MERGE_TEMPLATE = """\
### model
model_name_or_path: {model_name_or_path}
adapter_name_or_path: {adapter_path}
trust_remote_code: true
finetuning_type: lora
template: {template}

### export
export_dir: {export_dir}
export_size: 5
export_device: cpu
export_legacy_format: false
export_hub_model_id: {hub_model_id}
"""

        def _adapter_path(lora_out_rel, variant_name):
            best_file = Path(BASE_DIR) / lora_out_rel.lstrip("../") / "BEST_CHECKPOINT"
            if not best_file.exists():
                print(f"[SKIP] [{model_key}/{ds_name}][{variant_name}] BEST_CHECKPOINT not found at {best_file}. "
                      f"Run stage2 evaluation first.")
                return None, None
            name = best_file.read_text(encoding="utf-8").strip()
            return f"{lora_out_rel}/{name}", f"Stage 2 winner: {name}"

        adapter_base, selection_base = _adapter_path(cfg["save_s2_base"], "merge_base")
        adapter_world, selection_world = _adapter_path(cfg["save_s2_world"], "merge_world_model")

        if adapter_base is None or adapter_world is None:
            _skipped += 1
            continue

        MERGE_MODELS = {
            "base": {
                "model_name_or_path": cfg["model_id"],
                "adapter_path": adapter_base,
                "template": cfg["template"],
                "export_dir": cfg["out_s2_merged_base"],
                "hub_model_id": cfg["hf_s2_base"],
                "_selection": selection_base,
            },
            "world-model": {
                "model_name_or_path": cfg["out_s1_merged"],
                "adapter_path": adapter_world,
                "template": cfg["template"],
                "export_dir": cfg["out_s2_merged_world"],
                "hub_model_id": cfg["hf_s2_world"],
                "_selection": selection_world,
            },
        }

        print(f"=== {model_key} / {ds_name} Stage 2 Merge YAML ===")
        for fname, params in MERGE_MODELS.items():
            yaml_params = {k: v for k, v in params.items() if not k.startswith("_")}
            content = MERGE_TEMPLATE.format(**yaml_params)
            fpath = yaml_dir / f"{model_key}_{fname}.yaml"
            with open(fpath, "w") as f:
                f.write(content)
            print(f"[Created] {fpath.relative_to(Path(LF_ROOT))}")
            print(f"  base:      {params['model_name_or_path']}")
            print(f"  adapter:   {params['adapter_path']}  ({params['_selection']})")
            print(f"  export:    {params['export_dir']}")
            print(f"  hub_id:    {params['hub_model_id']}")
            print()
        _created += 1

print(f"--- Stage 2 Merge YAML: {_created} created, {_skipped} skipped ---")
if _skipped > 0:
    print("Re-run this cell after completing stage2 evaluation for the skipped models.")


In [ ]:
os.chdir(LF_ROOT)

## [Qwen2-VL-2B] [MobiBench] [stage2] Merge - Base
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-MB/stage2_merge/qwen2-vl-2b_base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2-VL-2B] [AndroidControl] [stage2] Merge - Base
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-AC/stage2_merge/qwen2-vl-2b_base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2-VL-7B] [MobiBench] [stage2] Merge - Base
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-MB/stage2_merge/qwen2-vl-7b_base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2-VL-7B] [AndroidControl] [stage2] Merge - Base
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-AC/stage2_merge/qwen2-vl-7b_base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2.5-VL-3B] [MobiBench] [stage2] Merge - Base
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-MB/stage2_merge/qwen2.5-vl-3b_base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2.5-VL-3B] [AndroidControl] [stage2] Merge - Base
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-AC/stage2_merge/qwen2.5-vl-3b_base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2.5-VL-7B] [MobiBench] [stage2] Merge - Base
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-MB/stage2_merge/qwen2.5-vl-7b_base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2.5-VL-7B] [AndroidControl] [stage2] Merge - Base
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-AC/stage2_merge/qwen2.5-vl-7b_base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen3-VL-2B] [MobiBench] [stage2] Merge - Base
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-MB/stage2_merge/qwen3-vl-2b_base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen3-VL-2B] [AndroidControl] [stage2] Merge - Base
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-AC/stage2_merge/qwen3-vl-2b_base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen3-VL-4B] [MobiBench] [stage2] Merge - Base
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-MB/stage2_merge/qwen3-vl-4b_base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen3-VL-4B] [AndroidControl] [stage2] Merge - Base
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-AC/stage2_merge/qwen3-vl-4b_base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen3-VL-8B] [MobiBench] [stage2] Merge - Base
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-MB/stage2_merge/qwen3-vl-8b_base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen3-VL-8B] [AndroidControl] [stage2] Merge - Base
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-AC/stage2_merge/qwen3-vl-8b_base.yaml

In [ ]:
os.chdir(BASE_DIR)

## [Gemma-4-E2B] [MobiBench] [stage2] Merge - Base & World Model | Unsloth
# NOTE: stage2_merge.sh 는 두 variant (base, world_model) 중 BEST_CHECKPOINT 가 있는 것만 머지합니다.
#       이 셀 실행 시점엔 base variant 만 BEST 가 있다고 가정합니다.
!bash "$BASE_DIR/scripts/stage2_merge.sh" --model gemma-4-e2b --dataset MB


In [ ]:
os.chdir(BASE_DIR)

## [Gemma-4-E2B] [AndroidControl] [stage2] Merge - Base & World Model | Unsloth
# NOTE: stage2_merge.sh 는 두 variant (base, world_model) 중 BEST_CHECKPOINT 가 있는 것만 머지합니다.
#       이 셀 실행 시점엔 base variant 만 BEST 가 있다고 가정합니다.
!bash "$BASE_DIR/scripts/stage2_merge.sh" --model gemma-4-e2b --dataset AC


In [ ]:
os.chdir(BASE_DIR)

## [Gemma-4-E4B] [MobiBench] [stage2] Merge - Base & World Model | Unsloth
# NOTE: stage2_merge.sh 는 두 variant (base, world_model) 중 BEST_CHECKPOINT 가 있는 것만 머지합니다.
#       이 셀 실행 시점엔 base variant 만 BEST 가 있다고 가정합니다.
!bash "$BASE_DIR/scripts/stage2_merge.sh" --model gemma-4-e4b --dataset MB


In [ ]:
os.chdir(BASE_DIR)

## [Gemma-4-E4B] [AndroidControl] [stage2] Merge - Base & World Model | Unsloth
# NOTE: stage2_merge.sh 는 두 variant (base, world_model) 중 BEST_CHECKPOINT 가 있는 것만 머지합니다.
#       이 셀 실행 시점엔 base variant 만 BEST 가 있다고 가정합니다.
!bash "$BASE_DIR/scripts/stage2_merge.sh" --model gemma-4-e4b --dataset AC


In [ ]:
os.chdir(LF_ROOT)

## [LLaVA-v1.6-Mistral-7B] [MobiBench] [stage2] Merge - Base
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-MB/stage2_merge/llava-v1.6-mistral-7b_base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [LLaVA-v1.6-Mistral-7B] [AndroidControl] [stage2] Merge - Base
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-AC/stage2_merge/llava-v1.6-mistral-7b_base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [LLaVA-v1.6-Vicuna-7B] [MobiBench] [stage2] Merge - Base
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-MB/stage2_merge/llava-v1.6-vicuna-7b_base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [LLaVA-v1.6-Vicuna-7B] [AndroidControl] [stage2] Merge - Base
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-AC/stage2_merge/llava-v1.6-vicuna-7b_base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [LLaMA3-LLaVA-Next-8B] [MobiBench] [stage2] Merge - Base
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-MB/stage2_merge/llama3-llava-next-8b_base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [LLaMA3-LLaVA-Next-8B] [AndroidControl] [stage2] Merge - Base
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-AC/stage2_merge/llama3-llava-next-8b_base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2-VL-2B] [MobiBench] [stage1+stage2] Merge - World Model
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-MB/stage2_merge/qwen2-vl-2b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2-VL-2B] [AndroidControl] [stage1+stage2] Merge - World Model
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-AC/stage2_merge/qwen2-vl-2b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2-VL-7B] [MobiBench] [stage1+stage2] Merge - World Model
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-MB/stage2_merge/qwen2-vl-7b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2-VL-7B] [AndroidControl] [stage1+stage2] Merge - World Model
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-AC/stage2_merge/qwen2-vl-7b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2.5-VL-3B] [MobiBench] [stage1+stage2] Merge - World Model
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-MB/stage2_merge/qwen2.5-vl-3b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2.5-VL-3B] [AndroidControl] [stage1+stage2] Merge - World Model
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-AC/stage2_merge/qwen2.5-vl-3b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2.5-VL-7B] [MobiBench] [stage1+stage2] Merge - World Model
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-MB/stage2_merge/qwen2.5-vl-7b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen2.5-VL-7B] [AndroidControl] [stage1+stage2] Merge - World Model
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-AC/stage2_merge/qwen2.5-vl-7b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen3-VL-2B] [MobiBench] [stage1+stage2] Merge - World Model
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-MB/stage2_merge/qwen3-vl-2b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen3-VL-2B] [AndroidControl] [stage1+stage2] Merge - World Model
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-AC/stage2_merge/qwen3-vl-2b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen3-VL-4B] [MobiBench] [stage1+stage2] Merge - World Model
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-MB/stage2_merge/qwen3-vl-4b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen3-VL-4B] [AndroidControl] [stage1+stage2] Merge - World Model
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-AC/stage2_merge/qwen3-vl-4b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen3-VL-8B] [MobiBench] [stage1+stage2] Merge - World Model
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-MB/stage2_merge/qwen3-vl-8b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [Qwen3-VL-8B] [AndroidControl] [stage1+stage2] Merge - World Model
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-AC/stage2_merge/qwen3-vl-8b_world-model.yaml

In [ ]:
os.chdir(BASE_DIR)

## [Gemma-4-E2B] [MobiBench] [stage1+stage2] Merge - Base & World Model | Unsloth
# NOTE: stage2_merge.sh 는 두 variant 중 BEST_CHECKPOINT 가 있는 것만 머지합니다.
#       이 셀 실행 시점에는 world_model variant 의 BEST 도 존재하여 함께 머지됩니다.
!bash "$BASE_DIR/scripts/stage2_merge.sh" --model gemma-4-e2b --dataset MB


In [ ]:
os.chdir(BASE_DIR)

## [Gemma-4-E2B] [AndroidControl] [stage1+stage2] Merge - Base & World Model | Unsloth
# NOTE: stage2_merge.sh 는 두 variant 중 BEST_CHECKPOINT 가 있는 것만 머지합니다.
#       이 셀 실행 시점에는 world_model variant 의 BEST 도 존재하여 함께 머지됩니다.
!bash "$BASE_DIR/scripts/stage2_merge.sh" --model gemma-4-e2b --dataset AC


In [ ]:
os.chdir(BASE_DIR)

## [Gemma-4-E4B] [MobiBench] [stage1+stage2] Merge - Base & World Model | Unsloth
# NOTE: stage2_merge.sh 는 두 variant 중 BEST_CHECKPOINT 가 있는 것만 머지합니다.
#       이 셀 실행 시점에는 world_model variant 의 BEST 도 존재하여 함께 머지됩니다.
!bash "$BASE_DIR/scripts/stage2_merge.sh" --model gemma-4-e4b --dataset MB


In [ ]:
os.chdir(BASE_DIR)

## [Gemma-4-E4B] [AndroidControl] [stage1+stage2] Merge - Base & World Model | Unsloth
# NOTE: stage2_merge.sh 는 두 variant 중 BEST_CHECKPOINT 가 있는 것만 머지합니다.
#       이 셀 실행 시점에는 world_model variant 의 BEST 도 존재하여 함께 머지됩니다.
!bash "$BASE_DIR/scripts/stage2_merge.sh" --model gemma-4-e4b --dataset AC


In [ ]:
os.chdir(LF_ROOT)

## [LLaVA-v1.6-Mistral-7B] [MobiBench] [stage1+stage2] Merge - World Model
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-MB/stage2_merge/llava-v1.6-mistral-7b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [LLaVA-v1.6-Mistral-7B] [AndroidControl] [stage1+stage2] Merge - World Model
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-AC/stage2_merge/llava-v1.6-mistral-7b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [LLaVA-v1.6-Vicuna-7B] [MobiBench] [stage1+stage2] Merge - World Model
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-MB/stage2_merge/llava-v1.6-vicuna-7b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [LLaVA-v1.6-Vicuna-7B] [AndroidControl] [stage1+stage2] Merge - World Model
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-AC/stage2_merge/llava-v1.6-vicuna-7b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [LLaMA3-LLaVA-Next-8B] [MobiBench] [stage1+stage2] Merge - World Model
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-MB/stage2_merge/llama3-llava-next-8b_world-model.yaml

In [ ]:
os.chdir(LF_ROOT)

## [LLaMA3-LLaVA-Next-8B] [AndroidControl] [stage1+stage2] Merge - World Model
!DISABLE_VERSION_CHECK=1 llamafactory-cli export examples/custom/GUI-Model-AC/stage2_merge/llama3-llava-next-8b_world-model.yaml

### 8. Stage 2 Evaluation Pipeline

**흐름: train → eval → merge.** eval 은 HF Hub 의존 없이 로컬 per-epoch checkpoint 만 소비한다.

1. **Baseline (zero-shot)** — `{model_id}` 에 대해 `vllm_infer.py` 로 `outputs/{DS}/eval/{MODEL}/stage2_eval/base/` 생성.
2. **Checkpoint sweep — 두 variant** (`--stage1-mode` 가 world-model 상류 모드를 결정):
   - `lora_base`: `--model_name_or_path {model_id} --adapter_name_or_path outputs/{DS}/adapters/{MODEL}_stage2_lora_base/epoch-*`
   - `lora_world_model_from_${MODE}`: `--model_name_or_path outputs/{DS}/merged/{MODEL}_stage1_${MODE}/` (로컬 merged, 없으면 hard-fail) `--adapter_name_or_path outputs/{DS}/adapters/{MODEL}_stage2_lora_world_model_from_${MODE}/epoch-*`
   - `max_lora_rank`: MB=16, AC=32
   - 결과: `outputs/{DS}/eval/{MODEL}/stage2_eval/{variant}/checkpoint-N/(generated_predictions.jsonl|action_metrics.json)`
3. **Winner 선정** — `_action_eval.py select` 가 `step_accuracy` 최댓값 checkpoint 를 각 adapter 디렉토리의 `BEST_CHECKPOINT[.json]` 에 기록. 이어지는 `stage2_merge.sh` 가 이 파일을 소비한다.

쉘 경로는 `bash scripts/stage2_eval.sh --model {MODEL} --dataset {DS} --stage1-mode {full|lora}`. Step Accuracy 정의: `ARCHITECTURE.md` §6.

> **HF Hub sweep**: `SaFD-00/{short}-{slug}stage2-{base|{mode}-world-model}-epoch{E}` 를 각 epoch 별로 pull 해 `vllm_infer.py` 호출. 쉘 경로: `bash scripts/stage2_eval.sh --model {MODEL} --dataset {DS} --stage1-mode {full|lora}`.


In [ ]:
os.chdir(LF_ROOT)

for model_key, ds_configs in CONFIGS.items():
    for ds_name, cfg in ds_configs.items():
        short_name = cfg["short_name"]
        _template = cfg["template"]
        _enable_thinking = "--enable_thinking False" if _template.startswith("qwen3_vl") else ""

        ## Base Zero-shot Prediction
        _out = f"../outputs/{cfg['output_prefix'].rstrip('/')}/eval/{short_name}/stage2_eval/base"
        !mkdir -p {_out}
        !python scripts/vllm_infer.py \
            --model_name_or_path {cfg["model_id"]} \
            --dataset {cfg['ds_s2_test']} \
            --template {_template} --cutoff_len 10000 --image_max_pixels {cfg["model_config"]["image_max_pixels"]} {_enable_thinking} \
            --save_name {_out}/generated_predictions.jsonl \
            --matrix_save_name {_out}/predict_results.json

In [ ]:
## [stage2] Prediction — HF Hub epoch 별 merged 모델 sweep (lora_base)
## epoch 리스트를 명시적으로 지정 (shell `stage2_eval.sh --epochs` 와 동일한 전환).
## 로컬 adapter 디렉토리는 조회하지 않고 HF Hub merged repo 만 pull 한다.
from pathlib import Path
os.chdir(LF_ROOT)

EPOCHS = [1, 2, 3]
HF_SLUG = {"MobiBench": "mb-", "AndroidControl": "ac-"}

for model_key, ds_configs in CONFIGS.items():
    for _ds_name, _cfg in ds_configs.items():
        if _cfg["model_config"].get("backend") == "unsloth":
            continue
        short_name = _cfg["short_name"]
        ds_code    = _cfg["output_prefix"].rstrip("/")
        _slug      = HF_SLUG[_ds_name]
        _template  = _cfg["template"]
        _enable_thinking = "--enable_thinking False" if _template.startswith("qwen3_vl") else ""
        _img_px    = _cfg["model_config"]["image_max_pixels"]
        _vllm_cfg  = '{"gpu_memory_utilization": 0.80}'   # merged 단일 모델
        _test_jsonl = f"{BASE_DIR}/data/{_ds_name}/gui-model_stage2_test.jsonl"

        _eval_dir_rel = f"../outputs/{ds_code}/eval/{short_name}/stage2_eval/lora_base"
        _eval_dir_abs = f"{LF_ROOT}/{_eval_dir_rel[3:]}"
        _adapter_dir  = Path(BASE_DIR) / "outputs" / ds_code / "adapters" / f"{short_name}_stage2_lora_base"
        _hub_template = f"SaFD-00/{short_name}-{_slug}stage2-base-epoch{{epoch}}"

        for epoch in EPOCHS:
            _hub_id = f"SaFD-00/{short_name}-{_slug}stage2-base-epoch{epoch}"
            _out = f"{_eval_dir_rel}/epoch-{epoch}"
            print(f"[{model_key}/{_ds_name}][lora_base] epoch-{epoch} → {_hub_id}")
            _cmd = (
                f"mkdir -p {_out} && python scripts/vllm_infer.py "
                f"--model_name_or_path {_hub_id} "
                f"--dataset {_cfg['ds_s2_test']} "
                f"--dataset_dir {LF_ROOT}/data "
                f"--template {_template} --cutoff_len 10000 "
                f"--image_max_pixels {_img_px} {_enable_thinking} "
                f"--vllm_config '{_vllm_cfg}' "
                f"--save_name {_out}/generated_predictions.jsonl "
                f"--matrix_save_name {_out}/predict_results.json"
            )
            !{_cmd}
            _score = (
                f"python {BASE_DIR}/scripts/_action_eval.py score "
                f"--test {_test_jsonl} "
                f"--pred {_out}/generated_predictions.jsonl "
                f"--output {_out}/action_metrics.json"
            )
            !{_score}

        _select = (
            f"python {BASE_DIR}/scripts/_action_eval.py select "
            f"--eval-dir {_eval_dir_abs} "
            f"--train-dir {_adapter_dir} "
            f"--metric step_accuracy "
            f"--hf-repo-template '{_hub_template}'"
        )
        !{_select}


In [ ]:
## [stage2] Prediction — HF Hub epoch 별 merged 모델 sweep (lora_world_model_from_{full,lora})
## epoch 리스트를 명시적으로 지정. HF Hub merged repo 가 world-model base 를 내장하므로
## 로컬 merged/{MODEL}_stage1_{MODE}/ 디렉토리 존재 여부와 무관하게 동작한다.
from pathlib import Path
os.chdir(LF_ROOT)

EPOCHS = [1, 2, 3]
HF_SLUG = {"MobiBench": "mb-", "AndroidControl": "ac-"}

for model_key, ds_configs in CONFIGS.items():
    for _ds_name, _cfg in ds_configs.items():
        if _cfg["model_config"].get("backend") == "unsloth":
            continue
        short_name = _cfg["short_name"]
        ds_code    = _cfg["output_prefix"].rstrip("/")
        _slug      = HF_SLUG[_ds_name]
        _template  = _cfg["template"]
        _enable_thinking = "--enable_thinking False" if _template.startswith("qwen3_vl") else ""
        _img_px    = _cfg["model_config"]["image_max_pixels"]
        _vllm_cfg  = '{"gpu_memory_utilization": 0.80}'
        _test_jsonl = f"{BASE_DIR}/data/{_ds_name}/gui-model_stage2_test.jsonl"

        for MODE in ("full", "lora"):
            _eval_dir_rel = f"../outputs/{ds_code}/eval/{short_name}/stage2_eval/lora_world_model_from_{MODE}"
            _eval_dir_abs = f"{LF_ROOT}/{_eval_dir_rel[3:]}"
            _adapter_dir  = Path(BASE_DIR) / "outputs" / ds_code / "adapters" / f"{short_name}_stage2_lora_world_model_from_{MODE}"
            _hub_template = f"SaFD-00/{short_name}-{_slug}stage2-{MODE}-world-model-epoch{{epoch}}"

            for epoch in EPOCHS:
                _hub_id = f"SaFD-00/{short_name}-{_slug}stage2-{MODE}-world-model-epoch{epoch}"
                _out = f"{_eval_dir_rel}/epoch-{epoch}"
                print(f"[{model_key}/{_ds_name}][lora_world_model_from_{MODE}] epoch-{epoch} → {_hub_id}")
                _cmd = (
                    f"mkdir -p {_out} && python scripts/vllm_infer.py "
                    f"--model_name_or_path {_hub_id} "
                    f"--dataset {_cfg['ds_s2_test']} "
                    f"--dataset_dir {LF_ROOT}/data "
                    f"--template {_template} --cutoff_len 10000 "
                    f"--image_max_pixels {_img_px} {_enable_thinking} "
                    f"--vllm_config '{_vllm_cfg}' "
                    f"--save_name {_out}/generated_predictions.jsonl "
                    f"--matrix_save_name {_out}/predict_results.json"
                )
                !{_cmd}
                _score = (
                    f"python {BASE_DIR}/scripts/_action_eval.py score "
                    f"--test {_test_jsonl} "
                    f"--pred {_out}/generated_predictions.jsonl "
                    f"--output {_out}/action_metrics.json"
                )
                !{_score}

            _select = (
                f"python {BASE_DIR}/scripts/_action_eval.py select "
                f"--eval-dir {_eval_dir_abs} "
                f"--train-dir {_adapter_dir} "
                f"--metric step_accuracy "
                f"--hf-repo-template '{_hub_template}'"
            )
            !{_select}


In [ ]:
import json
import re
import sys
from collections import defaultdict

# Stage 2 Action Prediction evaluator (정본).
# scripts/_action_eval.py 와 글자 단위 동치를 유지한다.
# 메트릭: Step Accuracy (SA) — bounds 영구 부재 데이터셋용 정의.
#   correct = (parse_ok ∧ type==gt ∧ field_match(type))
# field_match: navigate_back/finish 항상 통과 · click/long_click 의 index ·
#              scroll.direction · open_app.params.app · input.params.text

# ── Action Parsing ───────────────────────────────────────────────────────
def parse_action(text):
    text = (text or "").strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except Exception:
            pass
    match = re.search(r'\{[^{}]*\}', text)
    if match:
        try:
            return json.loads(match.group())
        except Exception:
            pass
    return None


# ── Field 추출 헬퍼 (top-level + nested params 모두 지원) ────────────────
def _pval(action, key):
    if action is None:
        return None
    if key in action:
        return action[key]
    return (action.get('params') or {}).get(key)


def _norm(s):
    return str(s if s is not None else '').strip().lower()


# ── Step Accuracy 채점 ───────────────────────────────────────────────────
# field_match(type) 가 정의된 type 만 step_correct 로 인정. 그 외 type 은 False.
_FIELD_MATCH_TYPES = {
    'navigate_back', 'finish', 'click', 'long_click',
    'scroll', 'open_app', 'input',
}


def evaluate_single(gt_action, pred_action):
    result = {
        'parsed': pred_action is not None,
        'type_correct': False,
        'step_correct': False,
        'has_index_check': False,    # click / long_click
        'has_dir_check': False,      # scroll
        'has_app_check': False,      # open_app
        'has_text_check': False,     # input
    }
    if pred_action is None:
        return result

    gt_type = str(gt_action.get('type', '')).lower()
    pred_type = str(pred_action.get('type', '')).lower()
    result['type_correct'] = (gt_type == pred_type)
    if not result['type_correct']:
        return result

    # field_match
    if gt_type in ('navigate_back', 'finish'):
        result['step_correct'] = True
        return result

    if gt_type in ('click', 'long_click'):
        result['has_index_check'] = True
        result['step_correct'] = (
            str(gt_action.get('index')) == str(pred_action.get('index'))
        )
        return result

    if gt_type == 'scroll':
        result['has_dir_check'] = True
        result['step_correct'] = (
            _norm(_pval(gt_action, 'direction')) == _norm(_pval(pred_action, 'direction'))
        )
        return result

    if gt_type == 'open_app':
        result['has_app_check'] = True
        result['step_correct'] = (
            _norm(_pval(gt_action, 'app')) == _norm(_pval(pred_action, 'app'))
        )
        return result

    if gt_type == 'input':
        result['has_text_check'] = True
        result['step_correct'] = (
            _norm(_pval(gt_action, 'text')) == _norm(_pval(pred_action, 'text'))
        )
        return result

    # unknown type — type 만 일치해도 step 은 False (정책)
    return result


def evaluate_predictions(test_path, pred_path):
    with open(test_path, 'r') as f:
        gt_entries = [json.loads(line) for line in f if line.strip()]
    with open(pred_path, 'r') as f:
        pred_entries = [json.loads(line) for line in f if line.strip()]

    if len(gt_entries) != len(pred_entries):
        print(f"[warn] length mismatch: gt={len(gt_entries)} pred={len(pred_entries)} "
              f"→ truncating to {min(len(gt_entries), len(pred_entries))}", file=sys.stderr)

    results = []
    per_type = defaultdict(lambda: {
        'count': 0, 'type_correct': 0, 'step_correct': 0,
    })
    cond = {
        'index': {'n': 0, 'k': 0},   # click + long_click
        'dir':   {'n': 0, 'k': 0},   # scroll
        'app':   {'n': 0, 'k': 0},   # open_app
        'text':  {'n': 0, 'k': 0},   # input
    }

    for gt_entry, pred_entry in zip(gt_entries, pred_entries):
        gt_action = json.loads(gt_entry['messages'][-1]['value'])
        pred_text = pred_entry.get('predict', pred_entry.get('output', ''))
        pred_action = parse_action(pred_text)

        r = evaluate_single(gt_action, pred_action)
        gt_type = str(gt_action.get('type', 'unknown')).lower()
        r['gt_type'] = gt_type
        results.append(r)

        per_type[gt_type]['count'] += 1
        per_type[gt_type]['type_correct'] += int(r['type_correct'])
        per_type[gt_type]['step_correct'] += int(r['step_correct'])

        if r['has_index_check']:
            cond['index']['n'] += 1
            cond['index']['k'] += int(r['step_correct'])
        if r['has_dir_check']:
            cond['dir']['n'] += 1
            cond['dir']['k'] += int(r['step_correct'])
        if r['has_app_check']:
            cond['app']['n'] += 1
            cond['app']['k'] += int(r['step_correct'])
        if r['has_text_check']:
            cond['text']['n'] += 1
            cond['text']['k'] += int(r['step_correct'])

    total = len(results)
    parsed = sum(1 for r in results if r['parsed'])
    type_correct = sum(int(r['type_correct']) for r in results)
    step_correct = sum(int(r['step_correct']) for r in results)

    parse_rate = parsed / total if total else 0
    type_acc = type_correct / total if total else 0
    step_acc = step_correct / total if total else 0

    per_type_summary = {}
    for t, d in per_type.items():
        per_type_summary[t] = {
            'count':    d['count'],
            'type_acc': round(d['type_correct'] / d['count'] if d['count'] else 0, 4),
            'step_acc': round(d['step_correct'] / d['count'] if d['count'] else 0, 4),
        }

    macro_step = (sum(v['step_acc'] for v in per_type_summary.values()) /
                  len(per_type_summary)) if per_type_summary else 0

    def _ratio(c):
        return c['k'] / c['n'] if c['n'] else 0

    return {
        'total':                total,
        'parse_rate':           round(parse_rate, 4),
        'type_accuracy':        round(type_acc, 4),
        'step_accuracy':        round(step_acc, 4),
        'macro_step_accuracy':  round(macro_step, 4),
        'cond_index_acc':       round(_ratio(cond['index']), 4),
        'cond_dir_acc':         round(_ratio(cond['dir']),   4),
        'cond_app_acc':         round(_ratio(cond['app']),   4),
        'cond_text_acc':        round(_ratio(cond['text']),  4),
        'per_type':             per_type_summary,
    }

print("[Loaded] Step Accuracy evaluator: parse_action, evaluate_single, evaluate_predictions")


In [ ]:
import json
import pandas as pd
from pathlib import Path

all_s2_metrics = {}
all_s2_results = {}

for model_key, ds_configs in CONFIGS.items():
    for ds_name, cfg in ds_configs.items():
        short_name = cfg["short_name"]
        op = cfg["output_prefix"]
        s2_eval_dir = Path(BASE_DIR) / "outputs" / op.rstrip("/") / "eval" / short_name / "stage2_eval"
        test_path = Path(cfg["data_dir"]) / "gui-model_stage2_test.jsonl"

        MODEL_PREDS = {}
        base_pred = s2_eval_dir / "base" / "generated_predictions.jsonl"
        if base_pred.exists():
            MODEL_PREDS["Base (Zero-shot)"] = base_pred
        for variant in ("lora_base", "lora_world_model"):
            for p in sorted((s2_eval_dir / variant).glob("epoch-*/generated_predictions.jsonl"),
                            key=lambda x: int(x.parent.name.split("-")[1])):
                MODEL_PREDS[f"{variant}/{p.parent.name}"] = p

        combo_key = f"{model_key}/{ds_name}"
        print(f"\n{'='*70}")
        print(f"=== {combo_key} Stage 2 Evaluation ({len(MODEL_PREDS)} model(s)) ===")

        ds_metrics = {}
        ds_results = {}
        for model_name, pred_path in MODEL_PREDS.items():
            if not pred_path.exists():
                continue
            metrics, results = evaluate_predictions(str(test_path), str(pred_path))
            ds_metrics[model_name] = metrics
            ds_results[model_name] = results
            out_metric_path = pred_path.parent / "action_metrics.json"
            with open(out_metric_path, 'w', encoding='utf-8') as f:
                json.dump({**metrics, "per_type": metrics.get("per_type", {})}, f, ensure_ascii=False, indent=2)
            print(f"[Done] {combo_key}/{model_name}  overall={metrics['overall_score']:.4f}")

        all_s2_metrics[combo_key] = ds_metrics
        all_s2_results[combo_key] = ds_results

        if ds_metrics:
            summary_df = pd.DataFrame({
                name: {
                    'Parse Rate': f"{m['parse_rate']:.1%}",
                    'Type Accuracy': f"{m['type_accuracy']:.1%}",
                    'Bounds IoU': f"{m['avg_bounds_iou']:.3f}",
                    'Params Accuracy': f"{m['params_accuracy']:.1%}",
                    'Cond. IoU': f"{m['cond_bounds_iou']:.3f}",
                    'Cond. Params': f"{m['cond_params_acc']:.1%}",
                    'Overall Score': f"{m['overall_score']:.3f}",
                } for name, m in ds_metrics.items()
            }).T
            print(f"\n=== Stage 2 Model Comparison - {combo_key} ===")
            display(summary_df)

            for variant in ("lora_base", "lora_world_model"):
                ckpt_metrics = {k.split("/")[1]: v for k, v in ds_metrics.items()
                                if k.startswith(f"{variant}/checkpoint-")}
                if not ckpt_metrics:
                    continue
                best_name, best_m = max(ckpt_metrics.items(),
                                        key=lambda kv: (kv[1]['overall_score'],
                                                        int(kv[0].split('-')[1])))
                lora_dir_rel = cfg["save_s2_base"] if variant == "lora_base" else cfg["save_s2_world"]
                lora_dir = Path(BASE_DIR) / lora_dir_rel.lstrip("../")
                lora_dir.mkdir(parents=True, exist_ok=True)
                (lora_dir / "BEST_CHECKPOINT").write_text(best_name + "\n", encoding='utf-8')
                summary = {
                    "selected": best_name,
                    "metric": "overall_score",
                    "metric_value": best_m['overall_score'],
                    "candidates": [
                        {"checkpoint": k, "overall_score": v['overall_score'],
                         "type_accuracy": v['type_accuracy'],
                         "avg_bounds_iou": v['avg_bounds_iou'],
                         "params_accuracy": v['params_accuracy'],
                         "parse_rate": v['parse_rate']}
                        for k, v in ckpt_metrics.items()
                    ],
                }
                (lora_dir / "BEST_CHECKPOINT.json").write_text(
                    json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding='utf-8')
                print(f"  [{combo_key}/{variant}] winner: {best_name} (overall={best_m['overall_score']:.4f})")
                print(f"     -> {lora_dir / 'BEST_CHECKPOINT'}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

for combo_key, ds_metrics in all_s2_metrics.items():
    if not ds_metrics:
        continue
    model_key, ds_name = combo_key.split("/")
    cfg = CONFIGS[model_key][ds_name]
    short_name = cfg["short_name"]
    model_names = list(ds_metrics.keys())
    metrics_to_plot = ['type_accuracy', 'avg_bounds_iou', 'params_accuracy', 'overall_score']
    metric_labels = ['Type Accuracy', 'Bounds IoU', 'Params Accuracy', 'Overall Score']
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))

    x = np.arange(len(metrics_to_plot))
    n_models = len(model_names)
    width = 0.8 / n_models
    colors = ['#9E9E9E', '#FF5722', '#2196F3', '#4CAF50'][:n_models]
    for i, name in enumerate(model_names):
        values = [ds_metrics[name][m] for m in metrics_to_plot]
        offset = (i - n_models / 2 + 0.5) * width
        axes[0].bar(x + offset, values, width, label=name, color=colors[i])
    axes[0].set_xlabel('Metric'); axes[0].set_ylabel('Score')
    axes[0].set_title(f'Stage 2: Model Comparison ({combo_key})')
    axes[0].set_xticks(x); axes[0].set_xticklabels(metric_labels, rotation=15)
    axes[0].legend(fontsize=8); axes[0].set_ylim(0, 1.0); axes[0].grid(axis='y', alpha=0.3)

    action_types = sorted(set(t for m in ds_metrics.values() for t in m['per_type'].keys()))
    x2 = np.arange(len(action_types))
    for i, name in enumerate(model_names):
        values = [ds_metrics[name]['per_type'].get(t, {}).get('type_acc', 0) for t in action_types]
        offset = (i - n_models / 2 + 0.5) * width
        axes[1].bar(x2 + offset, values, width, label=name, color=colors[i])
    axes[1].set_xlabel('Action Type'); axes[1].set_ylabel('Type Accuracy')
    axes[1].set_title(f'Per-Type Accuracy ({combo_key})')
    axes[1].set_xticks(x2); axes[1].set_xticklabels(action_types, rotation=30, fontsize=8)
    axes[1].legend(fontsize=8); axes[1].set_ylim(0, 1.0); axes[1].grid(axis='y', alpha=0.3)

    plt.tight_layout()
    op = cfg["output_prefix"]
    s2_eval_dir = Path(BASE_DIR) / "outputs" / op.rstrip("/") / "eval" / short_name / "stage2_eval"
    save_path = str(s2_eval_dir / "stage2_evaluation.png")
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"[Saved] {save_path}")

In [ ]:
import json
from pathlib import Path
from datetime import datetime

def generate_eval_report(model_name, metrics, results, output_dir, ds_name, cfg, comparison_metrics=None):
    m = metrics
    s2 = cfg["stage2"]
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    lines = []
    lines.append(f"# Stage 2 Evaluation Report: {model_name} ({cfg['model_key']}/{ds_name})")
    lines.append(f"\n> Generated: {now}\n")
    lines.append("## Experiment Setup\n")
    lines.append("| Item | Value |")
    lines.append("|------|-------|")
    lines.append(f"| Model | {cfg['model_id']} |")
    lines.append(f"| Dataset | {ds_name} |")
    lines.append(f"| Test Samples | {m['total']} |")
    lines.append(f"| Method | LoRA (r={s2['lora_rank']}, a={s2['lora_alpha']}, dropout={s2['lora_dropout']}) |")
    lines.append(f"| Training | {s2['epochs']} epochs, LR={s2['lr']} (cosine) |")
    lines.append("")
    lines.append("## Overall Metrics\n")
    lines.append("| Metric | Score |")
    lines.append("|--------|-------|")
    lines.append(f"| Parse Rate | {m['parse_rate']:.1%} |")
    lines.append(f"| Type Accuracy | {m['type_accuracy']:.1%} |")
    lines.append(f"| Bounds IoU (avg) | {m['avg_bounds_iou']:.4f} |")
    lines.append(f"| Bounds IoU (cond.) | {m['cond_bounds_iou']:.4f} |")
    lines.append(f"| Params Accuracy (avg) | {m['params_accuracy']:.1%} |")
    lines.append(f"| Params Accuracy (cond.) | {m['cond_params_acc']:.1%} |")
    lines.append(f"| **Overall Score** | **{m['overall_score']:.4f}** |")
    lines.append("")
    lines.append("## Per-Type Breakdown\n")
    lines.append("| Action Type | Count | Type Acc | Avg IoU | Params Acc |")
    lines.append("|-------------|-------|----------|---------|------------|")
    for t, d in sorted(m['per_type'].items(), key=lambda x: -x[1]['count']):
        lines.append(f"| {t} | {d['count']} | {d['type_acc']:.1%} | {d['avg_iou']:.4f} | {d['params_acc']:.1%} |")
    lines.append("")
    if comparison_metrics:
        lines.append("## Model Comparison\n")
        lines.append("| Metric | " + " | ".join(comparison_metrics.keys()) + " |")
        lines.append("|--------|" + "|".join(["-------"] * len(comparison_metrics)) + "|")
        for label, key, fmt in [("Parse Rate","parse_rate",".1%"),("Type Accuracy","type_accuracy",".1%"),
            ("Bounds IoU","avg_bounds_iou",".4f"),("Cond. IoU","cond_bounds_iou",".4f"),
            ("Params Accuracy","params_accuracy",".1%"),("Cond. Params","cond_params_acc",".1%"),
            ("Overall Score","overall_score",".4f")]:
            values = []
            scores = [cm[key] for cm in comparison_metrics.values()]
            best = max(scores)
            for name, cm in comparison_metrics.items():
                val = cm[key]
                formatted = f"{val:{fmt}}"
                if val == best and len(set(scores)) > 1:
                    formatted = f"**{formatted}**"
                values.append(formatted)
            lines.append(f"| {label} | " + " | ".join(values) + " |")
        lines.append("")
    report = "\n".join(lines)
    output_path = Path(output_dir) / "evaluation_report.md"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(report)
    return str(output_path)

for combo_key, ds_metrics in all_s2_metrics.items():
    model_key, ds_name = combo_key.split("/")
    cfg = CONFIGS[model_key][ds_name]
    short_name = cfg["short_name"]
    op = cfg["output_prefix"]
    s2_eval_dir = Path(BASE_DIR) / "outputs" / op.rstrip("/") / "eval" / short_name / "stage2_eval"
    OUTPUT_DIRS = {
        "Base (Zero-shot)": s2_eval_dir / "base",
        "stage2": s2_eval_dir / "lora_base",
        "stage1+stage2": s2_eval_dir / "lora_world_model",
    }
    for model_name, m in ds_metrics.items():
        output_dir = OUTPUT_DIRS.get(model_name)
        if output_dir:
            saved = generate_eval_report(model_name, m, all_s2_results[combo_key][model_name],
                                          output_dir, ds_name, cfg, comparison_metrics=ds_metrics)
            print(f"[Saved] {saved}")
print("\nDone.")